# Floor Plan Process Map (Layout-Based Process Mining)

Bird's-eye view of the factory: flows between work centers and between workstations are drawn directly on the floor plan (SVG).

1. **Section A** — infrastructure code (geometry, routing engine, floor-plan overlay, analysis). Run once; no need to modify.
2. **Section B** — edit the *Settings* cell for your dataset (file path, column names, hierarchy, floor-plan zones/doors).
3. **Section C** — run: load data, floor-plan diagrams (frequency & performance), analysis tables.

In [2]:
from __future__ import annotations

import colorsys
import heapq
import html
import io
import itertools
import math
import os
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from benchmark_01 import run_benchmark
from IPython.display import HTML, display


## Section A — Infrastructure

### A1. Geometry and formatting helpers

In [3]:
def _segment_intersects_rect(p0, p1, rect, margin=3.0):
    x0, y0 = p0
    x1, y1 = p1
    rx, ry, rw, rh = rect
    xmin, ymin = rx - margin, ry - margin
    xmax, ymax = rx + rw + margin, ry + rh + margin
    dx, dy = x1 - x0, y1 - y0
    tmin, tmax = 0.0, 1.0
    for p, q in ((-dx, x0 - xmin), (dx, xmax - x0), (-dy, y0 - ymin), (dy, ymax - y0)):
        if abs(p) < 1e-9:
            if q < 0:
                return False
            continue
        t = q / p
        if p < 0:
            if t > tmax:
                return False
            tmin = max(tmin, t)
        else:
            if t < tmin:
                return False
            tmax = min(tmax, t)
    return tmin <= tmax


def manhattan_fallback(exit_a, exit_b):
    _, y0 = exit_a
    x1, _ = exit_b
    mid = (x1, y0)
    if mid == exit_a or mid == exit_b:
        return [exit_a, exit_b]
    return [exit_a, mid, exit_b]


def fmt_dur(sec):
    sec = abs(float(sec))
    if sec < 60:
        return f"{sec:.0f} sec"
    if sec < 3600:
        return f"{sec / 60:.0f} min"
    if sec < 86400:
        return f"{sec / 3600:.1f} hr"
    return f"{sec / 86400:.1f} day"


def duration_to_color(value, vmin, vmax):
    if vmax <= vmin:
        t = 0.5
    else:
        t = min(max((value - vmin) / (vmax - vmin), 0.0), 1.0)
    green, yellow, red = (46, 204, 113), (241, 196, 15), (231, 76, 60)
    if t < 0.5:
        t2 = t / 0.5
        r = int(green[0] + (yellow[0] - green[0]) * t2)
        g = int(green[1] + (yellow[1] - green[1]) * t2)
        b = int(green[2] + (yellow[2] - green[2]) * t2)
    else:
        t2 = (t - 0.5) / 0.5
        r = int(yellow[0] + (red[0] - yellow[0]) * t2)
        g = int(yellow[1] + (red[1] - yellow[1]) * t2)
        b = int(yellow[2] + (red[2] - yellow[2]) * t2)
    return f"#{r:02x}{g:02x}{b:02x}"

### A2. Floor Plan Route Engine (floorplan_routing)

In [4]:
# ============================================================================
# Floor plan engine
# Wall thickness, scale, zone definitions, etc., are all passed as parameters;
# diagnostic outputs are written to the `log` list instead of using `print`.
# Safe for use in multi-user environments.
# ============================================================================

EPS = 1e-6
BEND_PEN = 60.0                        # each 90° turn = 60 px length penalty
DOOR_BALANCE_PX = 0.0                  # >0: door usage penalty (distributes congestion to doors)
_OPP = {"N": "S", "S": "N", "E": "W", "W": "E"}
_DIRV = {"E": (1, 0), "W": (-1, 0), "S": (0, 1), "N": (0, -1)}   # SVG: y down, x right
_DNAMES = "EWSN"                       # indices: E0 W1 S2 N3  (reverse direction = idx ^ 1)
_NORMAL = {"N": (0, -1), "S": (0, 1), "W": (-1, 0), "E": (1, 0)}
_VEC2DIR = {(0, -1): "N", (0, 1): "S", (-1, 0): "W", (1, 0): "E"}
_NUM = r'-?\d*\.?\d+(?:[eE][-+]?\d+)?'

# ============================================================================
# 1) SVG DOM: transform chain, area boxes, labels, id -> bbox
# ============================================================================
def _local(tag):
    return tag.split("}")[-1] if isinstance(tag, str) else ""


def _parse_transform(tstr):
    """Turns 3x3 matrix into a 3x3 affine matrix (rotate/translate/scale/matrix)."""
    M = np.eye(3)
    for name, args in re.findall(r'(\w+)\s*\(([^)]*)\)', tstr or ""):
        a = [float(v) for v in re.findall(_NUM, args)]
        T = np.eye(3)
        if name == "translate" and a:
            T[0, 2], T[1, 2] = a[0], (a[1] if len(a) > 1 else 0.0)
        elif name == "scale" and a:
            T[0, 0], T[1, 1] = a[0], (a[1] if len(a) > 1 else a[0])
        elif name == "rotate" and a:
            r = math.radians(a[0])
            c, s = math.cos(r), math.sin(r)
            R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])
            if len(a) >= 3:
                T1 = np.eye(3)
                T1[0, 2], T1[1, 2] = a[1], a[2]
                T2 = np.eye(3)
                T2[0, 2], T2[1, 2] = -a[1], -a[2]
                T = T1 @ R @ T2
            else:
                T = R
        elif name == "matrix" and len(a) == 6:
            T = np.array([[a[0], a[2], a[4]], [a[1], a[3], a[5]], [0, 0, 1.0]])
        M = M @ T
    return M


def _style_fill(el):
    f = el.get("fill")
    if f:
        return f.strip().lower()
    m = re.search(r"fill\s*:\s*([^;]+)", el.get("style", ""))
    return m.group(1).strip().lower() if m else ""


def _raw_bbox(el):
    t = _local(el.tag)
    if t == "rect":
        try:
            return (float(el.get("x", 0)), float(el.get("y", 0)),
                    float(el.get("width")), float(el.get("height")))
        except (TypeError, ValueError):
            return None
    if t == "line":
        try:
            x1, y1, x2, y2 = (float(el.get(k, 0)) for k in ("x1", "y1", "x2", "y2"))
        except (TypeError, ValueError):
            return None
        return (min(x1, x2), min(y1, y2), abs(x2 - x1), abs(y2 - y1))
    src = el.get("points") if t in ("polygon", "polyline") else el.get("d") if t == "path" else None
    if not src:
        return None
    nums = [float(v) for v in re.findall(_NUM, src)]
    xs, ys = nums[0::2], nums[1::2]
    n = min(len(xs), len(ys))
    xs, ys = xs[:n], ys[:n]
    return (min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)) if xs and ys else None


def _apply_M(bb, M):
    x, y, w, h = bb
    P = np.array([[x, y, 1], [x + w, y, 1], [x, y + h, 1], [x + w, y + h, 1]], float) @ M.T
    return (float(P[:, 0].min()), float(P[:, 1].min()),
            float(P[:, 0].max() - P[:, 0].min()), float(P[:, 1].max() - P[:, 1].min()))


def parse_svg_dom(svg_text, min_size=12.0, max_canvas_frac=0.6, is_area=None):
    """Applies the <g transform> chain to (1) area boxes + <text> labels,
    (2) the ROOT coordinates of each rect/line/polygon/path with an id.
    Returns: (nodes, id_bboxes, (canvas_w, canvas_h)).
    Default area definition: rect/polygon/path with a fill other than 'none', not too small/large.
    For custom rules, provide is_area(el, bbox) -> bool."""
    if re.search(r"<!ENTITY", svg_text, re.IGNORECASE):
        raise ValueError("there is a <!ENTITY> definition in the SVG; it has been rejected for security reasons.")
    try:
        root = ET.fromstring(svg_text.encode("utf-8"))
    except ET.ParseError as e:
        raise ValueError(f"SVG could not be parsed: {e}") from e
    vb = root.get("viewBox")
    try:
        if vb:
            cw, ch = [float(v) for v in vb.replace(",", " ").split()[2:4]]
        else:
            width_attr = root.get("width")
            height_attr = root.get("height")
            if width_attr is None or height_attr is None:
                raise ValueError("SVG width/height is missing.")
            cw = float(re.sub(r"[^\d.]", "", width_attr))
            ch = float(re.sub(r"[^\d.]", "", height_attr))
    except (TypeError, ValueError, IndexError) as e:
        raise ValueError("SVG viewBox / width-height could not be read.") from e
    nodes, texts, id_bboxes = [], [], {}

    def walk(el, M):
        if el.get("transform"):
            M = M @ _parse_transform(el.get("transform"))
        t = _local(el.tag)
        if t in ("rect", "polygon", "polyline", "path", "line"):
            bb = _raw_bbox(el)
            if bb:
                bb = _apply_M(bb, M)
                if el.get("id"):
                    id_bboxes[el.get("id")] = bb
                if t in ("rect", "polygon", "path"):
                    ok = (bb[2] >= min_size and bb[3] >= min_size
                          and bb[2] * bb[3] < max_canvas_frac * cw * ch
                          and _style_fill(el) != "none")
                    if is_area is not None:
                        ok = ok and is_area(el, bb)
                    if ok:
                        nodes.append({"bbox": bb, "id": el.get("id"), "labels": [], "area": bb[2] * bb[3]})
        elif t == "text":
            tsp = [c for c in el.iter() if _local(c.tag) == "tspan" and c.get("x") is not None]
            if tsp:
                items = [(" ".join("".join(c.itertext()).split()), c) for c in tsp]
            else:
                items = [(" ".join("".join(el.itertext()).split()), el)]
            for s, src in items:
                if s:
                    try:
                        px = float(src.get("x", 0) or 0) + 1.0
                        py = float(src.get("y", 0) or 0) - 3.0
                    except ValueError:
                        continue
                    p = M @ np.array([px, py, 1.0])
                    texts.append((s, (float(p[0]), float(p[1]))))
        for c in el:
            walk(c, M)

    walk(root, np.eye(3))
    # --- containment tree: parent = the smallest LARGER box that contains the center
    for i, a in enumerate(nodes):
        cx, cy = a["bbox"][0] + a["bbox"][2] / 2, a["bbox"][1] + a["bbox"][3] / 2
        cands = [j for j, b in enumerate(nodes)
                 if j != i and b["area"] > a["area"] * 1.001
                 and b["bbox"][0] <= cx <= b["bbox"][0] + b["bbox"][2]
                 and b["bbox"][1] <= cy <= b["bbox"][1] + b["bbox"][3]]
        a["parent"] = min(cands, key=lambda j: nodes[j]["area"]) if cands else None

    def depth(i):
        d = 0
        while nodes[i]["parent"] is not None:
            i = nodes[i]["parent"]
            d += 1
        return d

    for i, a in enumerate(nodes):
        a["depth"] = depth(i)
    # --- labels: are written in the SMALLEST box that contains every point of the <text>
    for s, (px, py) in texts:
        cands = [a for a in nodes if a["bbox"][0] <= px <= a["bbox"][0] + a["bbox"][2]
                 and a["bbox"][1] <= py <= a["bbox"][1] + a["bbox"][3]]
        if cands:
            min(cands, key=lambda a: a["area"])["labels"].append(s)
    return nodes, id_bboxes, (cw, ch)


def dom_positions(nodes, level_codes, fallback=None, log=None):
    """level_codes: [[level0 codes], [level1 codes], ...].
    Returns: (positions[level]{code:bbox}, ids[level]{code:svg_id}, missing[level][code...]).
    Level = the containment depth of the box (mode of the depths of candidate boxes); ID is not considered.
    For codes that cannot be found by labeling, `fallback(level, missing)` (ID-based fallback) is called."""
    log = log if log is not None else []
    out_pos, out_ids, out_missing = [], [], []
    for lvl, codes in enumerate(level_codes):
        pats = {c: re.compile(rf"(?<![\w-]){re.escape(str(c))}(?![\w-])", re.IGNORECASE) for c in codes}
        hits = [(nd, c) for nd in nodes for c, p in pats.items() if any(p.search(l) for l in nd["labels"])]
        pos, ids = {}, {}
        if hits:
            depths = [nd["depth"] for nd, _ in hits]
            mode = max(set(depths), key=depths.count)
            for nd, c in hits:
                if nd["depth"] == mode and (c not in pos or nd["area"] > pos[c][2] * pos[c][3]):
                    pos[c], ids[c] = nd["bbox"], nd["id"]
            log.append(f"Level {lvl}: containment depth={mode}, {len(pos)}/{len(codes)} codes matched from DOM")
        missing = [c for c in codes if c not in pos]
        if missing and fallback is not None:
            fb_pos, fb_ids = fallback(lvl, missing)
            if fb_pos:
                pos.update(fb_pos)
                ids.update(fb_ids)
                log.append(f"Level {lvl}: {len(fb_pos)} codes did not match labels, found with ID fallback "
                           f"(level information should be checked)")
        out_pos.append(pos)
        out_ids.append(ids)
        out_missing.append([c for c in codes if c not in pos])
    return out_pos, out_ids, out_missing


# ============================================================================
# 2) Factory border / wall / door geometry
# ============================================================================
def build_factory_geometry(id_bboxes, zones, wall_t):
    """zones: {name: {"border_id": str, "door_ids": [str, ...]}}.
    Returns: ({zone: {'bbox','doors','walls'}}, warnings).
    Walls = the 4 sides of the border rectangle, with door intervals removed."""
    geo, warns = {}, []
    for name, z in (zones or {}).items():
        bb = id_bboxes.get(z.get("border_id"))
        if bb is None:
            warns.append(f"'{z.get('border_id')}' could not be found, {name} zone skipped.")
            continue
        bx, by, bw, bh = bb
        doors = []
        for did in z.get("door_ids", []):
            db = id_bboxes.get(did)
            if db is None:
                warns.append(f"{name}: door '{did}' could not be found.")
                continue
            dx, dy, dw, dh = db
            dcx, dcy = dx + dw / 2, dy + dh / 2
            dist = {"N": abs(dcy - by), "S": abs(dcy - (by + bh)),
                    "W": abs(dcx - bx), "E": abs(dcx - (bx + bw))}
            side = min(dist, key=lambda d: dist[d])
            if side in "NS":
                lo, hi, line, c = dx, dx + dw, (by if side == "N" else by + bh), dcx
            else:
                lo, hi, line, c = dy, dy + dh, (bx if side == "W" else bx + bw), dcy
            doors.append({"id": did, "side": side, "lo": lo, "hi": hi, "line": line, "center": c})
        walls = []
        for side in "NSWE":
            ds = sorted((d["lo"], d["hi"]) for d in doors if d["side"] == side)
            if side in "NS":
                a0, a1, line = bx, bx + bw, (by if side == "N" else by + bh)
            else:
                a0, a1, line = by, by + bh, (bx if side == "W" else bx + bw)
            segs, cur = [], a0
            for lo, hi in ds:
                if lo > cur:
                    segs.append((cur, lo))
                cur = max(cur, hi)
            if cur < a1:
                segs.append((cur, a1))
            for s0, s1 in segs:
                if side in "NS":
                    walls.append((s0, line - wall_t / 2, s1 - s0, wall_t))
                else:
                    walls.append((line - wall_t / 2, s0, wall_t, s1 - s0))
        geo[name] = {"bbox": bb, "doors": doors, "walls": walls}
    return geo, warns


def zone_of_point(pt, geo):
    for name, g in (geo or {}).items():
        x, y, w, h = g["bbox"]
        if x < pt[0] < x + w and y < pt[1] < y + h:
            return name
    return None


def door_passage(door, lane_off, off):
    """Door passage points. For orthogonal passage, both points are on the same lane."""
    nx, ny = _NORMAL[door["side"]]
    c = door["center"] + lane_off
    p = (c, door["line"]) if door["side"] in "NS" else (door["line"], c)
    return (p[0] - nx * off, p[1] - ny * off), (p[0] + nx * off, p[1] + ny * off)


def _pick_doors(A, B, zA, zB, geo, used, door_off, balance_px=None):
    """The most shortest (Manhattan) door-door pair. If equal length, the door closest to the source is selected."""
    balance_px = DOOR_BALANCE_PX if balance_px is None else balance_px

    def man(p, q):
        return abs(p[0] - q[0]) + abs(p[1] - q[1])

    docsA = geo[zA]["doors"] if zA in geo else []
    docsB = geo[zB]["doors"] if zB in geo else []
    best = None
    for dA in (docsA or [None]):
        for dB in (docsB or [None]):
            chain, tie = [A], 0.0
            if dA:
                i, o = door_passage(dA, 0.0, door_off)
                chain += [i, o]
                tie += man(A, i)
            if dB:
                i, o = door_passage(dB, 0.0, door_off)
                chain += [o, i]
                tie += man(B, i)
            chain.append(B)
            cost = sum(man(p, q) for p, q in itertools.pairwise(chain))
            cost += balance_px * sum(used.get(d["id"], 0) for d in (dA, dB) if d)
            cost += 1e-3 * tie
            if best is None or cost < best[0]:
                best = (cost, dA, dB)
    if best is None:
        return None, None
    return best[1], best[2]


# ---- real distance (meter)
def _center_of(bbox):
    x, y, w, h = bbox
    return (x + w / 2.0, y + h / 2.0)


def door_path_length_px(pA, zA, pB, zB, geo):
    """Vertical-horizontal (forklift/transportation) distance; shortest door(s) across different zones."""
    def man(p, q):
        return abs(p[0] - q[0]) + abs(p[1] - q[1])

    if zA == zB or not geo:
        return man(pA, pB)
    docsA = geo[zA]["doors"] if zA in geo else []
    docsB = geo[zB]["doors"] if zB in geo else []
    best = math.inf
    for dA in (docsA or [None]):
        for dB in (docsB or [None]):
            chain = [pA]
            if dA:
                chain.append(door_passage(dA, 0.0, 0.0)[0])
            if dB:
                chain.append(door_passage(dB, 0.0, 0.0)[0])
            chain.append(pB)
            best = min(best, sum(man(p, q) for p, q in itertools.pairwise(chain)))
    return best


def layout_distance(a, b, positions, geo, px_to_m):
    """Returns: (straight-line m, door-based m, zone_a, zone_b)."""
    ca, cb = _center_of(positions[a]), _center_of(positions[b])
    za, zb = zone_of_point(ca, geo), zone_of_point(cb, geo)
    eu = math.hypot(cb[0] - ca[0], cb[1] - ca[1])
    walk = door_path_length_px(ca, za, cb, zb, geo)
    return eu * px_to_m, walk * px_to_m, za, zb


# ============================================================================
# 3) Level style (line width, arrow, font... derived from box width)
# ============================================================================
def derive_level_style(positions, ref_stat="max", overrides=None):
    ov = dict(overrides or {})
    thick = np.array([min(w, h) for (_x, _y, w, h) in positions.values() if w > 0 and h > 0], dtype=float)
    if thick.size == 0:
        u = 50.0
    elif ref_stat == "max":
        u = float(thick.max())
    elif ref_stat == "p90":
        u = float(np.percentile(thick, 90))
    elif ref_stat == "median":
        u = float(np.median(thick))
    else:
        raise ValueError("ref_stat: 'max' | 'p90' | 'median'")

    def clamp(v, lo, hi):
        return max(lo, min(hi, v))

    st = {
        "ref": u,
        "max_width": clamp(u * 0.06, 1.2, 10.0),
        "min_width": clamp(u * 0.02, 0.5, 2.0),
        "font_size": clamp(u * 0.12, 6.0, 16.0),
        "arrow_min": clamp(u * 0.05, 3.0, 10.0),
        "arrow_k": 3.2,
        "arrow_half_ratio": 0.45,
        "loop_size": clamp(u * 0.10, 6.0, 40.0),
        "loop_gap": clamp(u * 0.08, 4.0, 30.0),
        "label_halo": None,
        "title_font": None,
        "cell_size": None,
        "corridor_margin": None,
        "buffer_radius": None,
        "buffer_weight": 0.35,
        "port_gap_min": None,
        "port_gap_comfort": None,
        "stub": None,
        "stub_step": None,
    }
    st.update(ov)
    if st["label_halo"] is None:
        st["label_halo"] = st["font_size"] * 0.35
    if st["title_font"] is None:
        st["title_font"] = clamp(st["font_size"] * 1.5, 10.0, 24.0)
    if st["cell_size"] is None:
        st["cell_size"] = int(clamp(round(u / 40.0), 2, 10))
    if st["corridor_margin"] is None:
        st["corridor_margin"] = max(clamp(u * 0.05, 3.0, 24.0), st["max_width"] * 0.75 + 1.0)
    if st["buffer_radius"] is None:
        st["buffer_radius"] = int(clamp(math.ceil(st["max_width"] / st["cell_size"]), 1, 3))
    cs = st["cell_size"]
    arrow_len_max = max(st["arrow_min"], st["arrow_k"] * st["max_width"])
    if st["port_gap_min"] is None:
        st["port_gap_min"] = max(3.2 * st["max_width"], 2.0 * cs, 4.0)
    if st["port_gap_comfort"] is None:
        st["port_gap_comfort"] = st["port_gap_min"] * 2.0
    if st["stub_step"] is None:
        st["stub_step"] = max(2.0 * cs, st["max_width"] * 2.0)
    if st["stub"] is None:
        st["stub"] = max(arrow_len_max * 1.4, st["corridor_margin"] + cs * 1.5)
    return st


# ============================================================================
# 4) Arrow / label / self-loop drawing
# ============================================================================
def _arrow_for(path, w, st):
    """path: world-coordinate path. Returns: (trimmed_path, arrow_polygon_points_str)."""
    L = max(st["arrow_min"], st["arrow_k"] * w)
    hw = L * st["arrow_half_ratio"]
    (x0, y0), (x1, y1) = path[-2], path[-1]
    dx, dy = x1 - x0, y1 - y0
    seg = math.hypot(dx, dy)
    ux, uy = (1.0, 0.0) if seg < 1e-9 else (dx / seg, dy / seg)
    bx, by = x1 - ux * L, y1 - uy * L
    if seg > L:
        trimmed = list(path[:-1]) + [(bx, by)]
    else:
        trimmed = list(path[:-1])
        if len(trimmed) < 2:
            trimmed = [path[0], (bx, by)]
    px, py = -uy, ux
    poly = (f"{x1:.1f},{y1:.1f} {bx + px * hw:.1f},{by + py * hw:.1f} "
            f"{bx - px * hw:.1f},{by - py * hw:.1f}")
    return trimmed, poly


def _label_anchor(path, w, st):
    """Places the label at the center of the longest segment of the path. Returns: (x, y, text_anchor)."""
    if not path:
        return 0.0, 0.0, "middle"
    if len(path) == 1:
        x, y = path[0]
        f = st["font_size"]
        return x, y - w / 2.0 - f * 0.3, "middle"

    best, best_len = None, -1.0
    for i in range(len(path) - 1):
        (xa, ya), (xb, yb) = path[i], path[i + 1]
        ln = math.hypot(xb - xa, yb - ya)
        if ln > best_len:
            best, best_len = (xa, ya, xb, yb), ln

    if best is None:
        x, y = path[0]
        f = st["font_size"]
        return x, y - w / 2.0 - f * 0.3, "middle"

    xa, ya, xb, yb = best
    mx, my = (xa + xb) / 2.0, (ya + yb) / 2.0
    f = st["font_size"]
    if abs(xb - xa) >= abs(yb - ya):
        return mx, my - w / 2.0 - f * 0.3, "middle"
    return mx + w / 2.0 + f * 0.3, my + f * 0.35, "start"


def _label_svg(x, y, anchor, text, st):
    return (f'<text x="{x:.1f}" y="{y:.1f}" font-family="Arial" font-size="{st["font_size"]:.1f}" '
            f'font-weight="bold" fill="#1B2631" text-anchor="{anchor}" '
            f'style="paint-order: stroke; stroke:#FFFFFF; stroke-width:{st["label_halo"]:.1f}px;">{text}</text>')


def _self_loop_geometry(bbox, cnt, st, stroke_w, label_text=None):
    x, y, w, _h = bbox
    L = max(st["arrow_min"], st["arrow_k"] * stroke_w)
    hw = L * st["arrow_half_ratio"]
    gap = max(min(st["loop_gap"], w * 0.30), stroke_w * 2.5)
    loop_h = max(st["loop_size"], L * 1.6)
    p_out = (x + w - gap * 2, y)
    p_in = (x + w - gap, y)
    mid = (p_out[0] + p_in[0]) / 2.0
    f = st["font_size"]
    label_y = y - 0.75 * loop_h - stroke_w / 2.0 - f * 0.35
    txt_hw = len(label_text or f"{cnt:,}") * f * 0.36 + st["label_halo"]
    pad = f * 0.3
    rx0 = min(p_out[0] - hw, mid - txt_hw) - pad
    rx1 = max(p_in[0] + hw, mid + txt_hw) + pad
    ry0 = min(y - loop_h - hw, label_y - f) - pad
    return {"L": L, "hw": hw, "gap": gap, "loop_h": loop_h, "p_out": p_out, "p_in": p_in,
            "mid": mid, "label_y": label_y, "rect": (rx0, ry0, rx1 - rx0, y - ry0)}


def draw_self_loop_svg(bbox, count, color, st, stroke_w, label_text=None):
    geo = _self_loop_geometry(bbox, count, st, stroke_w, label_text)
    y = bbox[1]
    L, hw, loop_h = geo["L"], geo["hw"], geo["loop_h"]
    p_out, p_in = geo["p_out"], geo["p_in"]
    end = (p_in[0], y - L)
    path = (f'<path d="M {p_out[0]:.1f},{p_out[1]:.1f} '
            f'C {p_out[0]:.1f},{y - loop_h:.1f} {p_in[0]:.1f},{y - loop_h:.1f} {end[0]:.1f},{end[1]:.1f}" '
            f'fill="none" stroke="{color}" stroke-width="{stroke_w:.2f}" '
            f'stroke-linecap="round" opacity="0.9"/>')
    arrow = (f'<polygon points="{p_in[0]:.1f},{p_in[1]:.1f} '
             f'{p_in[0] - hw:.1f},{y - L:.1f} {p_in[0] + hw:.1f},{y - L:.1f}" '
             f'fill="{color}" opacity="0.9"/>')
    label = _label_svg(geo["mid"], geo["label_y"], "middle", label_text or f"{count:,}", st)
    return path, arrow, label


def _loop_rect_for_side(bbox, cnt, st, stroke_w, side, label_text=None):
    rx, ry, rw, rh = _self_loop_geometry(bbox, cnt, st, stroke_w, label_text)["rect"]
    if side == "N":
        return (rx, ry, rw, rh)
    return (rx, bbox[1] + bbox[3], rw, rh)


def draw_self_loop_svg_side(bbox, count, color, st, stroke_w, side="N", label_text=None):
    lp, la, ll = draw_self_loop_svg(bbox, count, color, st, stroke_w, label_text)
    if side == "N":
        return lp, la, ll
    _x, y, _w, h = bbox
    geo = _self_loop_geometry(bbox, count, st, stroke_w, label_text)
    f = st["font_size"]
    label = _label_svg(geo["mid"], (2 * y + h) - geo["label_y"] + f * 0.75, "middle",
                       label_text or f"{count:,}", st)
    g = f'<g transform="translate(0,{2 * y + h:.2f}) scale(1,-1)">{lp}{la}</g>'
    return g, "", label


# ============================================================================
# 5) Ortogonal route helpers
# ============================================================================
def _orthogonalize(points):
    out = [points[0]]
    for p in points[1:]:
        q = out[-1]
        if abs(p[0] - q[0]) > 1e-6 and abs(p[1] - q[1]) > 1e-6:
            out.append((p[0], q[1]))
        out.append(p)
    return out


def _simplify_points(pts, eps=1e-6):
    """Removes duplicate and collinear intermediate points; leaves endpoints untouched."""
    dedup = []
    for p in pts:
        if dedup and abs(p[0] - dedup[-1][0]) < eps and abs(p[1] - dedup[-1][1]) < eps:
            continue
        dedup.append(p)
    if len(dedup) < 3:
        return dedup
    res = [dedup[0]]
    for i in range(1, len(dedup) - 1):
        a, b, c = res[-1], dedup[i], dedup[i + 1]
        cross = (b[0] - a[0]) * (c[1] - b[1]) - (b[1] - a[1]) * (c[0] - b[0])
        dot = (b[0] - a[0]) * (c[0] - b[0]) + (b[1] - a[1]) * (c[1] - b[1])
        if abs(cross) < eps and dot > 0:
            continue
        res.append(b)
    res.append(dedup[-1])
    return res


def _sdir(p, q):
    dx, dy = q[0] - p[0], q[1] - p[1]
    if abs(dx) > EPS and abs(dy) < EPS:
        return "E" if dx > 0 else "W"
    if abs(dy) > EPS and abs(dx) < EPS:
        return "S" if dy > 0 else "N"
    return None


def _path_cost(pts, d_start, d_end):
    """length + BEND_PEN*bends. Start/end directions must be perpendicular to the sides; backtracking is forbidden."""
    dirs = [d for d in (_sdir(p, q) for p, q in itertools.pairwise(pts)) if d]
    if not dirs:
        return math.inf, 0
    L = sum(abs(p[0] - q[0]) + abs(p[1] - q[1]) for p, q in itertools.pairwise(pts))
    bends, prev = 0, d_start
    for d in dirs:
        if d != prev:
            bends += 1
            prev = d
    if dirs[-1] != d_end:
        bends += 1
    pen = (1e6 if dirs[0] == _OPP[d_start] else 0) + (1e6 if dirs[-1] == _OPP[d_end] else 0)
    return L + BEND_PEN * bends + pen, bends


def _direct_candidates(A, B):
    (x0, y0), (x1, y1) = A, B
    c = [[A, B]] if (abs(x0 - x1) < EPS or abs(y0 - y1) < EPS) else []
    c += [[A, (x1, y0), B], [A, (x0, y1), B]]
    for f in (0.5, 0.25, 0.75):
        mx, my = x0 + (x1 - x0) * f, y0 + (y1 - y0) * f
        c += [[A, (mx, y0), (mx, y1), B], [A, (x0, my), (x1, my), B]]
    return c


def _port_xy(bb, side, t):
    x, y, w, h = bb
    return {"N": (t, y), "S": (t, y + h), "W": (x, t), "E": (x + w, t)}[side]


def _stub_xy(p, side, L):
    dx, dy = _DIRV[side]
    return (p[0] + dx * L, p[1] + dy * L)


def _side_candidates(Ca, Cb, tol=2.0):
    """Source sides based on the quadrant of the target center; target sides are their counterparts."""
    dx, dy = Cb[0] - Ca[0], Cb[1] - Ca[1]
    s = []
    if abs(dx) > tol:
        s.append("E" if dx > 0 else "W")
    if abs(dy) > tol:
        s.append("S" if dy > 0 else "N")
    if not s:
        s = ["E"]
    s.sort(key=lambda k: (k in "NS") if abs(dx) >= abs(dy) else (k in "EW"))
    return s, [_OPP[k] for k in s]


def _ideal_coord(side, bb, other, corner, anchor="center"):
    """Position of a port along a box side. If the two boxes overlap along the side's axis: centre of the overlap.
    Otherwise anchor="center" -> centre of the side (old behaviour); anchor="nearest" -> the point of the side
    closest to the other box (shorter, straighter links on wide boxes)."""
    x, y, w, h = bb
    X, Y, W, H = other
    if side in "NS":
        lo, hi, a0, a1, oc = max(x, X), min(x + w, X + W), x, x + w, X + W / 2
    else:
        lo, hi, a0, a1, oc = max(y, Y), min(y + h, Y + H), y, y + h, Y + H / 2
    k = corner * (2.0 if anchor == "nearest" else 1.0)
    if hi - lo > 2 * corner:
        c = (lo + hi) / 2
    elif anchor == "nearest" and a1 - a0 > 2 * k:
        c = oc
    else:
        c = (a0 + a1) / 2
        k = corner
    return min(max(c, a0 + k), a1 - k)


def box_crossings(points, boxes, shrink=1.0):
    out = []
    for k, (p, q) in enumerate(itertools.pairwise(points)):
        for bi, r in enumerate(boxes):
            if r[2] > 2 * shrink and r[3] > 2 * shrink and _segment_intersects_rect(p, q, r, margin=-shrink):
                out.append((k, bi))
    return out


def count_wall_crossings(points, wall_rects):
    """Counts the number of segments that intersect with walls outside the doors."""
    return sum(any(_segment_intersects_rect(points[i], points[i + 1], r, margin=0.0) for r in wall_rects)
               for i in range(len(points) - 1))


# ---- numpy collision
def _arr(rects):
    return np.array(list(rects), float).reshape(-1, 4)


def _inf(a, m):
    return a + np.array([-m, -m, 2 * m, 2 * m]) if len(a) else a


def a_hit(p, q, arr, m=0.0):
    """if segment is parallel to an axis, checks if it intersects any rectangle in the array (tangent not counted)."""
    if len(arr) == 0:
        return False
    x0, x1 = (p[0], q[0]) if p[0] <= q[0] else (q[0], p[0])
    y0, y1 = (p[1], q[1]) if p[1] <= q[1] else (q[1], p[1])
    return bool(((x0 < arr[:, 0] + arr[:, 2] + m - EPS) & (x1 > arr[:, 0] - m + EPS) &
                 (y0 < arr[:, 1] + arr[:, 3] + m - EPS) & (y1 > arr[:, 1] - m + EPS)).any())


def a_path_hit(pts, arr, m=0.0):
    return any(a_hit(p, q, arr, m) for p, q in itertools.pairwise(pts))


def a_window(arr, x0, y0, x1, y1):
    if len(arr) == 0:
        return arr
    mk = (arr[:, 0] < x1) & (arr[:, 0] + arr[:, 2] > x0) & (arr[:, 1] < y1) & (arr[:, 1] + arr[:, 3] > y0)
    return arr[mk]


# ---- Hanan grid + bending-penalized Dijkstra (NumPy masked)
def r4_hanan(A, B, d_start, d_end, arr):
    rects = arr.tolist()
    xs = sorted({A[0], B[0]} | {v for r in rects for v in (r[0], r[0] + r[2])})
    ys = sorted({A[1], B[1]} | {v for r in rects for v in (r[1], r[1] + r[3])})
    nx, ny = len(xs), len(ys)
    xa, ya = np.array(xs), np.array(ys)
    vb = np.zeros((ny, nx), bool)
    hb = np.zeros((ny, max(nx - 1, 1)), bool)
    wb = np.zeros((max(ny - 1, 1), nx), bool)
    for rx, ry, rw, rh in rects:
        i0, i1 = int(np.searchsorted(xa, rx)), int(np.searchsorted(xa, rx + rw))
        j0, j1 = int(np.searchsorted(ya, ry)), int(np.searchsorted(ya, ry + rh))
        vb[j0 + 1:j1, i0 + 1:i1] = True
        hb[j0 + 1:j1, i0:i1] = True
        wb[j0:j1, i0 + 1:i1] = True
    vbl, hbl, wbl = vb.tolist(), hb.tolist(), wb.tolist()
    xi, yi = {x: i for i, x in enumerate(xs)}, {y: j for j, y in enumerate(ys)}
    si, sj, gi, gj = xi[A[0]], yi[A[1]], xi[B[0]], yi[B[1]]
    if vbl[sj][si] or vbl[gj][gi]:
        return None
    d0 = _DNAMES.index(d_start)
    de = _DNAMES.index(d_end)
    _h = lambda i, j: abs(xs[i] - xs[gi]) + abs(ys[j] - ys[gj]) 
    start = (si, sj, d0)
    best, prev, pq, cnt = {start: 0.0}, {}, [(_h(si, sj), 0, 0.0, start)], 0
    while pq:
        _, _, c, cur = heapq.heappop(pq)
        if best.get(cur, math.inf) < c - 1e-9:
            continue
        i, j, d = cur
        if (i, j) == (gi, gj):
            pts, key = [], cur
            while True:
                pts.append((xs[key[0]], ys[key[1]]))
                if key == start:
                    break
                key = prev[key]
            return _simplify_points(pts[::-1])
        for nd in range(4):
            if nd == (d ^ 1):
                continue
            if nd == 0:
                ok = i + 1 < nx and not hbl[j][i] and not vbl[j][i + 1]
                ni, nj = i + 1, j
            elif nd == 1:
                ok = i - 1 >= 0 and not hbl[j][i - 1] and not vbl[j][i - 1]
                ni, nj = i - 1, j
            elif nd == 2:
                ok = j + 1 < ny and not wbl[j][i] and not vbl[j + 1][i]
                ni, nj = i, j + 1
            else:
                ok = j - 1 >= 0 and not wbl[j - 1][i] and not vbl[j - 1][i]
                ni, nj = i, j - 1
            if not ok:
                continue
            nc = c + abs(xs[ni] - xs[i]) + abs(ys[nj] - ys[j]) + (BEND_PEN if nd != d else 0)
            if (ni, nj) == (gi, gj) and nd != de:
                nc += BEND_PEN
            key = (ni, nj, nd)
            if nc < best.get(key, math.inf) - 1e-9:
                best[key] = nc
                prev[key] = cur
                cnt += 1
                heapq.heappush(pq, (nc + _h(ni, nj), cnt, nc, key))
    return None


def r4_route_between(A, B, ds, de, arr, allow_hanan=True):
    if math.dist(A, B) < EPS:
        return [A]
    best = None
    for c in _direct_candidates(A, B):
        if not a_path_hit(c, arr):
            cost = _path_cost(c, ds, de)[0]
            if best is None or cost < best[0]:
                best = (cost, c)
    if best:
        return _simplify_points(best[1])
    if not allow_hanan:
        return None
    for pad in (200.0, 600.0, None):
        sub = arr if pad is None else a_window(arr, min(A[0], B[0]) - pad, min(A[1], B[1]) - pad,
                                               max(A[0], B[0]) + pad, max(A[1], B[1]) + pad)
        pts = r4_hanan(A, B, ds, de, sub)
        if pts and not a_path_hit(pts, arr):
            return pts
    return None


# ---- Door sprites / port distribution
def r4_lane(door, used, step):
    width = door["hi"] - door["lo"]
    offs = []
    for s in (step, step / 2.0, step / 4.0):
        n = max(1, int((width - 4.0) // s) + 1)
        for o in sorted(((i - (n - 1) / 2.0) * s for i in range(n)), key=abs):
            if all(abs(o - x) > 1e-6 for x in offs):
                offs.append(o)
    k = used.get(door["id"], 0)
    used[door["id"]] = k + 1
    return offs[k % len(offs)]


def r4_spread(ts, lo, hi, gap):
    n = len(ts)
    if n == 1:
        return [min(max(ts[0], lo), hi)]
    order = sorted(range(n), key=lambda i: ts[i])
    v = [ts[i] for i in order]
    for k in range(1, n):
        v[k] = max(v[k], v[k - 1] + gap)
    sh = (sum(ts) - sum(v)) / n
    v = [x + sh for x in v]
    if v[-1] > hi:
        d = v[-1] - hi
        v = [x - d for x in v]
    if v[0] < lo:
        d = lo - v[0]
        v = [x + d for x in v]
    if v[-1] > hi:
        v = [lo + (hi - lo) * k / (n - 1) for k in range(n)]
    out = [0.0] * n
    for pos, i in enumerate(order):
        out[i] = v[pos]
    return out


def r4_assign_lanes(entries, door, clearance):
    """entries: [(key, thickness, order_value)].
    Returns: (deviation from center {key: px}, width {key: px}, shrinkage_ratio)."""
    entries = sorted(entries, key=lambda e: e[2])
    n = len(entries)
    avail = (door["hi"] - door["lo"]) - 2 * clearance
    raw = sum(e[1] for e in entries)
    scale = 1.0 if raw + clearance * (n - 1) <= avail else max(0.05, (avail - clearance * (n - 1)) / raw)
    ws = [max(1.0, e[1] * scale) for e in entries]
    x = -(sum(ws) + clearance * (n - 1)) / 2.0
    offs, widths = {}, {}
    for e, w in zip(entries, ws):
        offs[e[0]] = x + w / 2.0
        widths[e[0]] = w
        x += w + clearance
    if n > 1 and sum(ws) + clearance * (n - 1) > avail:
        # door capacity exceeded: lines must still remain within the door opening (not extend to walls);
        # strips are distributed equally, remaining overlap is reported as 'OVERLAP' in the log.
        half = max(0.0, (avail - 1.0) / 2.0)
        for j, e in enumerate(entries):
            offs[e[0]] = -half + 2 * half * j / (n - 1)
    return offs, widths, scale


# ---- parallel strip separation
def r4_orient(p, q, eps=0.6):
    if abs(p[1] - q[1]) < eps and abs(p[0] - q[0]) > eps:
        return "H"
    if abs(p[0] - q[0]) < eps and abs(p[1] - q[1]) > eps:
        return "V"
    return None


def r4_try_move(pts, k, o, c, hard, min_end, apply=True):
    n = len(pts) - 1
    v0, v1 = pts[k], pts[k + 1]
    nv0 = (v0[0], c) if o == "H" else (c, v0[1])
    nv1 = (v1[0], c) if o == "H" else (c, v1[1])
    if a_hit(nv0, nv1, hard):
        return False
    for kk, pv, nv, far in ((k - 1, v0, nv0, pts[k - 1]), (k + 1, v1, nv1, pts[k + 2])):
        old = abs(far[0] - pv[0]) + abs(far[1] - pv[1])
        new = abs(far[0] - nv[0]) + abs(far[1] - nv[1])
        dot = (pv[0] - far[0]) * (nv[0] - far[0]) + (pv[1] - far[1]) * (nv[1] - far[1])
        if dot <= 0 or new < (min_end if kk in (0, n - 1) else 2.0):
            return False
        if new > old and a_hit(pv, nv, hard):
            return False
    if apply:
        pts[k], pts[k + 1] = nv0, nv1
    return True


def r4_measure(routes, near_px):
    H, V = defaultdict(list), defaultdict(list)
    for i, pts in enumerate(routes):
        for p, q in itertools.pairwise(pts):
            o = r4_orient(p, q)
            if o == "H":
                H[round(p[1], 1)].append((min(p[0], q[0]), max(p[0], q[0]), i))
            elif o == "V":
                V[round(p[0], 1)].append((min(p[1], q[1]), max(p[1], q[1]), i))
    exact = near = 0.0
    for D in (H, V):
        keys = sorted(D)
        for ia, a in enumerate(keys):
            for b in keys[ia:]:
                if b - a >= near_px:
                    break
                for s1 in D[a]:
                    for s2 in D[b]:
                        if s1[2] == s2[2] or (a == b and s1[2] > s2[2]):
                            continue
                        ln = min(s1[1], s2[1]) - max(s1[0], s2[0])
                        if ln > 1:
                            if b - a < 1.0:
                                exact += ln
                            else:
                                near += ln
    return exact, near


def r4_separate(routes, hard_for, fixed_arr, gap, min_end, iters=5):
    """Splits overlapping or closely spaced PARALLEL segments lying on the same line into strips.
    Port segments and segments passing through doors are fixed; they do not enter box/wall/door blockers.
    Returns: (number of moved segments, {'fixed':.., 'no_space':..})."""
    stats = {"fixed": 0, "no_space": 0}
    total = 0
    for _it in range(iters):
        fixedflag = [[a_hit(p[k], p[k + 1], fixed_arr) for k in range(len(p) - 1)] for p in routes]
        segs = []
        for ri, pts in enumerate(routes):
            n = len(pts) - 1
            ff = fixedflag[ri]
            for k in range(n):
                p, q = pts[k], pts[k + 1]
                o = r4_orient(p, q)
                if o is None:
                    continue
                c = p[1] if o == "H" else p[0]
                a_, b_ = sorted((p[0], q[0])) if o == "H" else sorted((p[1], q[1]))
                mv = 0 < k < n - 1 and not ff[k] and not ff[k - 1] and not ff[k + 1]
                segs.append({"r": ri, "k": k, "o": o, "c": c, "a": a_, "b": b_, "mv": mv})
        parent = list(range(len(segs)))

        def find(i, parent=parent):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i

        conflict = False
        for o in "HV":
            idx = sorted([i for i, s in enumerate(segs) if s["o"] == o], key=lambda i: segs[i]["c"])
            for ii, i in enumerate(idx):
                si = segs[i]
                for j in idx[ii + 1:]:
                    sj = segs[j]
                    if sj["c"] - si["c"] >= gap * 0.999:
                        break
                    if si["r"] != sj["r"] and min(si["b"], sj["b"]) - max(si["a"], sj["a"]) > 1.0 \
                            and (si["mv"] or sj["mv"]):
                        parent[find(i)] = find(j)
                        conflict = True
        if not conflict:
            break
        comps = defaultdict(list)
        for i in range(len(segs)):
            comps[find(i)].append(i)
        moved = 0
        for members in comps.values():
            mv = [i for i in members if segs[i]["mv"]]
            stats["fixed"] += len(members) - len(mv)
            if len(members) < 2 or not mv:
                continue
            o = segs[members[0]]["o"]
            placed = [(segs[i]["c"], segs[i]["a"], segs[i]["b"], segs[i]["r"]) for i in members if not segs[i]["mv"]]

            def sign(i, o=o):
                s = segs[i]  # noqa: B023
                pts = routes[s["r"]]
                k = s["k"]

                def sg(nb, cur):
                    d = (nb[1] - cur[1]) if o == "H" else (nb[0] - cur[0])
                    return -1 if d < -0.6 else (1 if d > 0.6 else 0)

                return sg(pts[k - 1], pts[k]) + sg(pts[k + 2], pts[k + 1])

            mv.sort(key=lambda i: (sign(i), segs[i]["c"], segs[i]["r"]))
            cm = float(np.mean([segs[i]["c"] for i in members]))
            for rank, i in enumerate(mv):
                s = segs[i]
                pts = routes[s["r"]]
                k = s["k"]
                p, q = pts[k], pts[k + 1]
                a_, b_ = sorted((p[0], q[0])) if o == "H" else sorted((p[1], q[1]))
                desired = cm + (rank - (len(mv) - 1) / 2.0) * gap
                hard = hard_for(s["r"])
                chosen = None
                for c in sorted((desired + t * gap * 0.5 for t in range(-40, 41)), key=lambda c: abs(c - desired)):
                    if any(pr != s["r"] and abs(c - pc) < gap - 1e-6 and min(b_, pb) - max(a_, pa) > 1.0
                           for pc, pa, pb, pr in placed):
                        continue
                    if r4_try_move(pts, k, o, c, hard, min_end):
                        chosen = c
                        break
                if chosen is None:
                    stats["no_space"] += 1
                    placed.append((s["c"], a_, b_, s["r"]))
                else:
                    placed.append((chosen, a_, b_, s["r"]))
                    moved += 1
        for ri in range(len(routes)):
            routes[ri] = _simplify_points(routes[ri])
        total += moved
        if moved == 0:
            break
    return total, stats


def r4_top_overlaps(routes, labels, fixed_arr, topn=12):
    """Diagnosis: longest overlaps. Returns: log lines."""
    segs = []
    for ri, pts in enumerate(routes):
        n = len(pts) - 1
        for k, (p, q) in enumerate(itertools.pairwise(pts)):
            o = r4_orient(p, q)
            if o is None:
                continue
            c = p[1] if o == "H" else p[0]
            a_, b_ = sorted((p[0], q[0])) if o == "H" else sorted((p[1], q[1]))
            role = "door" if a_hit(p, q, fixed_arr) else ("first" if k == 0 else ("last" if k == n - 1 else "middle"))
            segs.append((o, c, a_, b_, ri, role))
    res = []
    for o in "HV":
        S = sorted([s for s in segs if s[0] == o], key=lambda s: s[1])
        for i, s in enumerate(S):
            for t in S[i + 1:]:
                if t[1] - s[1] >= 1.0:
                    break
                if s[4] == t[4]:
                    continue
                ln = min(s[3], t[3]) - max(s[2], t[2])
                if ln > 1:
                    res.append((ln, o, s[1], labels[s[4]], s[5], labels[t[4]], t[5],
                                len(routes[s[4]]) - 1, len(routes[t[4]]) - 1))
    res.sort(reverse=True)
    pair = Counter()
    for r in res:
        pair[tuple(sorted((r[4], r[6])))] += r[0]
    out = []
    if pair:
        out.append("Overlap px, segment role pair: " + str({k: round(v) for k, v in pair.items()}))
    for ln, o, c, e1, r1, e2, r2, n1, n2 in res[:topn]:
        out.append(f"  {ln:6.0f}px {o} @{c:.0f} | {e1[0]}->{e1[1]} ({r1}, {n1} seg) x "
                   f"{e2[0]}->{e2[1]} ({r2}, {n2} seg)")
    return out


def r4_overlap_report(routes, fixed_arr, gap, geo, wt):
    """Diagnosis: remaining overlap classes + door capacity. Returns: log lines."""
    segs = []
    for ri, pts in enumerate(routes):
        n = len(pts) - 1
        for k, (p, q) in enumerate(itertools.pairwise(pts)):
            o = r4_orient(p, q)
            if o is None:
                continue
            c = p[1] if o == "H" else p[0]
            a_, b_ = sorted((p[0], q[0])) if o == "H" else sorted((p[1], q[1]))
            cls = "door" if a_hit(p, q, fixed_arr) else ("port" if k in (0, n - 1) else "middle")
            segs.append((o, c, a_, b_, ri, cls))
    tot = defaultdict(float)
    for o in "HV":
        S = sorted([s for s in segs if s[0] == o], key=lambda s: s[1])
        for i, s in enumerate(S):
            for t in S[i + 1:]:
                if t[1] - s[1] >= gap * 0.5:
                    break
                if s[4] == t[4] or abs(t[1] - s[1]) >= 1.0:
                    continue
                ln = min(s[3], t[3]) - max(s[2], t[2])
                if ln > 1:
                    cl = {s[5], t[5]}
                    tot["door" if "door" in cl else ("port" if "port" in cl else "middle")] += ln
    out = ["Remaining overlaps (px) class distribution: " + str({k: round(v) for k, v in tot.items()})]
    for g in geo.values():
        for d in g["doors"]:
            if d["side"] in "NS":
                r = np.array([[d["lo"], d["line"] - wt, d["hi"] - d["lo"], 2 * wt]])
            else:
                r = np.array([[d["line"] - wt, d["lo"], 2 * wt, d["hi"] - d["lo"]]])
            width = d["hi"] - d["lo"]
            cap = max(1, int((width - 4.0) // gap) + 1)
            use = sum(1 for pts in routes if a_path_hit(pts, r))
            out.append(f"  door {d['id']}: width={width:.0f}px | {gap:.0f}px spacing capacity={cap} lanes | "
                       f"lines passing={use}" + ("  <-- OVERFLOW" if use > cap else ""))
    return out


# ============================================================================
# 6) Interaction (hover / zoom / pan)
# because SVG is the first element in the iframe, so document.querySelector('svg') is used.
# ============================================================================
_JS_FLOORPLAN = """
(function() {
    var svg = document.querySelector('svg'),
        layer = svg.getElementById('edges'),
        hl = svg.getElementById('hl');
    var edges = [].slice.call(svg.querySelectorAll('.e')),
        tls = [].slice.call(svg.querySelectorAll('.tl')),
        nodes = [].slice.call(svg.querySelectorAll('.n')),
        byId = {};
    nodes.forEach(function(n) { byId[n.getAttribute('data-id')] = n; });
    function touches(el, id) {
        return el.getAttribute('data-s') === id || el.getAttribute('data-t') === id;
    }
    function enter(id) {
        edges.forEach(function(e) {
            if (touches(e, id)) {
                e.classList.add('hl');
                hl.appendChild(e);
                var o = e.getAttribute('data-s') === id ? e.getAttribute('data-t') : e.getAttribute('data-s');
                if (o !== id && byId[o]) byId[o].classList.add('p');
            } else e.classList.add('dim');
        });
        tls.forEach(function(t) { if (!touches(t, id)) t.classList.add('dim'); });
    }
    function leave() {
        edges.forEach(function(e) {
            e.classList.remove('hl');
            e.classList.remove('dim');
            layer.appendChild(e);
        });
        tls.forEach(function(t) { t.classList.remove('dim'); });
        nodes.forEach(function(n) { n.classList.remove('p'); });
    }
    nodes.forEach(function(n) {
        n.addEventListener('mouseenter', function() { enter(n.getAttribute('data-id')); });
        n.addEventListener('mouseleave', leave);
    });
    var vb = svg.viewBox.baseVal,
        X0 = vb.x, Y0 = vb.y, W0 = vb.width, H0 = vb.height;
    function pt(ev) {
        var p = svg.createSVGPoint();
        p.x = ev.clientX;
        p.y = ev.clientY;
        return p.matrixTransform(svg.getScreenCTM().inverse());
    }
    svg.addEventListener('wheel', function(ev) {
        ev.preventDefault();
        var k = ev.deltaY < 0 ? 0.85 : 1.18,
            p = pt(ev);
        vb.x = p.x - (p.x - vb.x) * k;
        vb.y = p.y - (p.y - vb.y) * k;
        vb.width *= k;
        vb.height *= k;
    }, { passive: false });
    var drag = false, lx = 0, ly = 0;
    svg.addEventListener('mousedown', function(ev) {
        drag = true;
        lx = ev.clientX;
        ly = ev.clientY;
    });
    window.addEventListener('mouseup', function() { drag = false; });
    window.addEventListener('mousemove', function(ev) {
        if (!drag) return;
        var r = svg.getBoundingClientRect(),
            s = Math.max(vb.width / r.width, vb.height / r.height);
        vb.x -= (ev.clientX - lx) * s;
        vb.y -= (ev.clientY - ly) * s;
        lx = ev.clientX;
        ly = ev.clientY;
    });
    svg.addEventListener('dblclick', function() {
        vb.x = X0; vb.y = Y0; vb.width = W0; vb.height = H0;
    });
})();
"""

_CSS_FLOORPLAN = """
.e .lbl { display:none }
.e.hl .lbl { display:inline }
.e.dim { opacity:.06 }
.e.hl {
	opacity:1;
	pointer-events:none
}
.tl.dim { opacity:.05 }
.n { cursor:pointer }
.n:hover {
	stroke:#1B2631;
	stroke-width:2.4;
	fill:#F7DC6F;
	fill-opacity:.18
}
.n.p {
	stroke:#E67E22;
	stroke-width:2.2;
	fill:#F5CBA7;
	fill-opacity:.30
}
"""


def _make_svg_responsive(content):
    """Root <svg>: if viewbox is not present, generate it from width/height, remove fixed width/height."""
    m = re.search(r'<svg\b[^>]*>', content)
    if not m:
        return content
    tag = m.group(0)
    new = tag
    if "viewBox" not in tag:
        w = re.search(r'\swidth="([\d.]+)', tag)
        h = re.search(r'\sheight="([\d.]+)', tag)
        if w and h:
            new = new.replace("<svg", f'<svg viewBox="0 0 {w.group(1)} {h.group(1)}"', 1)
    new = re.sub(r'\s(?:width|height)="[^"]*"', "", new)
    return content.replace(tag, new, 1)


# ============================================================================
# 7) MAIN OVERLAY: ports with perpendicular contact, 
# door/wall awareness, lane-based routing
# ============================================================================
def build_floor_plan_overlay(content, positions, pairs_df, *, title="", node_colors=None,
                             self_loop_counts=None, factory_geo=None, container_positions=None,
                             node_names=None, wall_t=8.0, min_count=1, top_n=None, label_top_n=40,
                             margin_px=None, style_overrides=None, ref_stat="max", self_loop_min_count=1,
                             mode="freq", perf_df=None, loop_perf=None, title_font=None, legend_font=None,
                             box_clearance=None, loop_clearance=None, side_search="all",
                             port_anchor="nearest", line_gap=None, sep_iters=5):
    """content: (painted) floor plan SVG text. Returns: (svg_text, info_dict).
    Raises ValueError if no edges to draw."""
    T0 = time.time()
    log = []
    st = derive_level_style(positions, ref_stat=ref_stat, overrides=style_overrides)
    # print(f"Level Style: ref={st['ref']:.1f}px | Line {st['min_width']:.2f}-{st['max_width']:.2f}px | "
    #       f"Font {st['font_size']:.1f}px | loop {st['loop_size']:.1f}/{st['loop_gap']:.1f}px | "
    #       f"ok >= {st['arrow_min']:.1f}px")    
    node_colors = node_colors or {}
    df_ = pairs_df[(pairs_df["count"] >= min_count)
                   & pairs_df["from"].isin(positions) & pairs_df["to"].isin(positions)].copy()
    if mode == "perf":
        if perf_df is None or perf_df.empty:
            raise ValueError("Performance mode requires perf_df.")
        df_ = df_.merge(perf_df[["from", "to", "mean_sec", "min_sec", "max_sec"]],
                        on=["from", "to"], how="left")    
    if top_n:
        df_ = df_.iloc[np.argsort(-df_["count"].to_numpy(), kind="stable")].head(top_n)
    if df_.empty:
        raise ValueError("No edges to draw.")

    def ctr(c):
        return _center_of(positions[c])

    df_["_d"] = [math.dist(ctr(a), ctr(b)) for a, b in zip(df_["from"], df_["to"])]
    df_ = df_.iloc[np.argsort(df_["_d"].to_numpy(), kind="stable")].reset_index(drop=True)

    # ---- duvar / kapı tıkacı / bölge
    geo = factory_geo or {}
    wt = wall_t
    walls = [w for g in geo.values() for w in g["walls"]]
    plugs, fixed_rects = [], []
    for g in geo.values():
        for d in g["doors"]:
            if d["side"] in "NS":
                plugs.append((d["lo"], d["line"] - wt / 2, d["hi"] - d["lo"], wt))
                fixed_rects.append((d["lo"], d["line"] - wt, d["hi"] - d["lo"], 2 * wt))
            else:
                plugs.append((d["line"] - wt / 2, d["lo"], wt, d["hi"] - d["lo"]))
                fixed_rects.append((d["line"] - wt, d["lo"], 2 * wt, d["hi"] - d["lo"]))
    zone = {c: zone_of_point(ctr(c), geo) for c in positions}

    leaf_rects = list(positions.values())
    loops = {c: n for c, n in (self_loop_counts or {}).items() if c in positions and n >= self_loop_min_count}
    max_count = max([df_["count"].max()] + list(loops.values()))
    loop_perf = loop_perf or {}
    if mode == "perf":
        _vals = list(df_["mean_sec"].dropna()) + [loop_perf[c] for c in loops if c in loop_perf]
        vmin, vmax = (np.percentile(_vals, [5, 95]) if _vals else (0.0, 1.0))
    else:
        vmin = vmax = 0.0

    def loop_label(c, n):
        return fmt_dur(loop_perf[c]) if mode == "perf" and c in loop_perf else f"{n:,}"

    def width_of(n):
        return round(st["min_width"] + (n / max_count) * (st["max_width"] - st["min_width"]), 2)

    arrow_len = max(st["arrow_min"], st["arrow_k"] * st["max_width"])
    # box_clearance: the HARD minimum distance (px) between any line and any box. The router first tries the
    # comfortable `margin`, then half of it, and finally this minimum. None -> old behaviour (1.5 px).
    min_clear = 1.5 if box_clearance is None else float(box_clearance)
    margin = margin_px if margin_px is not None else max(4.0, st["corridor_margin"] * 0.6)
    margin = max(margin, min_clear)
    margins = tuple(dict.fromkeys(max(m, min_clear) for m in (margin, margin * 0.5, min_clear)))
    hard_clear = 1.0 if box_clearance is None else min_clear      # used by the strip-separation step
    wall_m = max(1.5, st["max_width"] / 2.0 + 1.5)
    corner = max(2.0, st["max_width"])
    # line_gap: centre-to-centre spacing (px) between parallel lines; used for spreading ports along a box side
    # and for splitting overlapping parallel segments. None -> auto (1.5 x max line width + 3).
    gap = float(line_gap) if line_gap is not None else max(1.5 * st["max_width"] + 3.0, 5.0)
    door_off = wt / 2 + margins[0] + 2.0
    min_end = arrow_len * 1.1

    # ---- Top boxes (floor plan position based) and self-loop rectangles
    cont = [((j, c), tuple(bb)) for j, d in (container_positions or {}).items() for c, bb in d.items()]

    def anc(code):
        cx, cy = ctr(code)
        return frozenset(k for k, r in cont if r[0] < cx < r[0] + r[2] and r[1] < cy < r[1] + r[3])

    ancs = {c: anc(c) for c in positions}
    cont_raw, cont_keys = _arr([r for _, r in cont]), [k for k, _ in cont]

    # ---- self-loop edges: find a side (N/S) that does not intersect any other box or wall, 
    # and compute the rectangle for the loop label
    # Self-loop placement: try every (side, horizontal shift) candidate and keep the first one whose WHOLE
    # footprint (curve + arrow + label) touches no other box, wall or door. The footprint is inflated by
    # `loop_clearance`. Candidates are tried nearest-to-the-corner first; N before S. If nothing is free, the
    # candidate with the fewest collisions wins (the old code silently kept the last tried side, "S").
    # door approach corridors: lines must be able to reach every door, so loops keep out of them
    door_reach = door_off + max(arrow_len * 1.2, margins[0] + 3.0) + 2.0 * st["max_width"]
    door_zones = []
    for g in geo.values():
        for d in g["doors"]:
            if d["side"] in "NS":
                door_zones.append((d["lo"], d["line"] - door_reach, d["hi"] - d["lo"], 2 * door_reach))
            else:
                door_zones.append((d["line"] - door_reach, d["lo"], 2 * door_reach, d["hi"] - d["lo"]))
    loop_clear = loop_clearance if loop_clearance is not None else max(3.0, st["max_width"] * 0.5)
    loop_side, loop_rects, loop_bb = {}, {}, {}
    for c, n in loops.items():
        bx_, by_, bw_, bh_ = positions[c]
        obst = _inf(_arr([r for k, r in positions.items() if k != c] + walls + door_zones), loop_clear)
        lbl_, sw_ = loop_label(c, n), width_of(n)
        step = max(st["arrow_min"], st["font_size"] * 0.5)
        max_shift = max(0.0, bw_ - 4.0 * st["loop_gap"])
        shifts = [i * step for i in range(int(max_shift // step) + 1)]

        def n_hit(rect, obst=obst):
            if not len(obst):
                return 0
            return int(((obst[:, 0] < rect[0] + rect[2]) & (obst[:, 0] + obst[:, 2] > rect[0]) &
                        (obst[:, 1] < rect[1] + rect[3]) & (obst[:, 1] + obst[:, 3] > rect[1])).sum())

        best = None
        for sh in shifts:
            bb = (bx_, by_, bw_ - sh, bh_)
            for side in ("N", "S"):
                rect = _loop_rect_for_side(bb, n, st, sw_, side, lbl_)
                k = n_hit(rect)
                if best is None or k < best[0]:
                    best = (k, side, bb, rect, sh)
                if k == 0:
                    break
            if best[0] == 0:
                break
        loop_side[c], loop_bb[c], loop_rects[c] = best[1], best[2], best[3]
        if best[0]:
            log.append(f"Self-loop {c}: no collision-free position, {best[0]} collision(s) remain")
        elif best[4]:
            log.append(f"Self-loop {c}: shifted {best[4]:.0f}px left, side {best[1]}")

    box_raw = _arr(leaf_rects + list(loop_rects.values()))
    wall_raw = _arr(walls + plugs)
    wall_block = _inf(wall_raw, wall_m)

    def stub_ok(P, A, own_zone):
        """Port->stub piece must not intersect with walls/doors and the stub end must remain within its own zone."""
        if len(wall_block) and a_hit(P, A, wall_block):
            return False
        return zone_of_point(A, geo) == own_zone

    cache = {}

    def obs(mg, allowed):
        ck = (mg, allowed)
        if ck not in cache:
            parts = [_inf(box_raw, mg), _inf(wall_raw, wall_m)]
            if allowed is not None and len(cont_raw):
                mk = np.array([k not in allowed for k in cont_keys])
                parts.append(_inf(cont_raw[mk], mg))
            cache[ck] = np.vstack([p for p in parts if len(p)])
        return cache[ck]

    lane_off, lane_w = {}, {}
    cur: dict[str, int | None] = {"i": None}

    def lane_for(d, used):
        o = lane_off.get((cur["i"], d["id"]))
        return o if o is not None else r4_lane(d, used, st["max_width"] + 2.5)

    # ---- zoned leg routing (door-to-door transitions)
    def zoned(A, B, ds, de, za, zb, arr, used, allow_hanan):
        def leg(P, Q, d0, d1):
            return r4_route_between(P, Q, d0, d1, arr, allow_hanan)

        if za == zb or not geo:
            return leg(A, B, ds, de)
        dA, dB = _pick_doors(A, B, za, zb, geo, used, door_off)
        wps, pairs = [[A, None, ds]], set()
        if dA:
            vA = _NORMAL[dA["side"]]
            nd = _VEC2DIR[(vA[0], vA[1])]
            i, o = door_passage(dA, lane_for(dA, used), door_off)
            wps.append([i, nd, None])
            pairs.add(len(wps) - 1)
            wps.append([o, None, nd])
        if dB:
            vB = _NORMAL[dB["side"]]
            nd = _VEC2DIR[(vB[0], vB[1])]
            i, o = door_passage(dB, lane_for(dB, used), door_off)
            wps.append([o, _OPP[nd], None])
            pairs.add(len(wps) - 1)
            wps.append([i, None, _OPP[nd]])
        wps.append([B, de, None])
        pts = [A]
        for k in range(len(wps) - 1):
            P, Q = wps[k], wps[k + 1]
            if k in pairs:
                pts.append(Q[0])
                continue
            seg = leg(P[0], Q[0], P[2], Q[1])
            if seg is None:
                return None
            pts.extend(seg[1:])
        return pts

    def route_edge(a, sa, ta, b, sb, tb, allow_hanan, used, use_forb=True):
        pa, pb = _port_xy(positions[a], sa, ta), _port_xy(positions[b], sb, tb)
        allowed = frozenset(ancs[a] | ancs[b]) if use_forb else None
        for mg in margins:
            arr = obs(mg, allowed)
            L = max(arrow_len * 1.2, mg + 3.0)
            A, B = _stub_xy(pa, sa, L), _stub_xy(pb, sb, L)
            if not (stub_ok(pa, A, zone[a]) and stub_ok(pb, B, zone[b])):
                continue          # This margin is breaking through the stub wall: try a margin with a shorter stub.
            u = dict(used)
            mid = zoned(A, B, sa, _OPP[sb], zone[a], zone[b], arr, u, allow_hanan)
            if mid:
                used.update(u)
                full = _simplify_points([pa] + mid + [pb])
                return full, _path_cost(full, sa, _OPP[sb])[0]
        return None

    # ================= 1. TRANSITION: selection of the front using only cheap (direct) candidates
    t1 = time.time()
    plan, tmp_used = [], {}
    for _, r in df_.iterrows():
        a, b, cnt = r["from"], r["to"], int(r["count"])
        srcs, dsts = _side_candidates(ctr(a), ctr(b))
        pref = [(s, t) for s in srcs for t in dsts]
        rest = [(s, t) for s in "NSWE" for t in "NSWE" if (s, t) not in pref]
        best = None
        # side_search="all": all 16 side pairs compete on real route cost (length + bends); the quadrant
        # heuristic only wins ties. "quadrant": old behaviour (quadrant pairs first, others only as a fallback).
        pref_set = set(pref)
        groups = (pref + rest,) if side_search == "all" else (pref, rest)
        for group in groups:
            for use_forb in (True, False):
                for s, t in group:
                    ta = _ideal_coord(s, positions[a], positions[b], corner, port_anchor)
                    tb = _ideal_coord(t, positions[b], positions[a], corner, port_anchor)
                    res = route_edge(a, s, ta, b, t, tb, False, tmp_used, use_forb)
                    if res:
                        cost = res[1] + (0.0 if (s, t) in pref_set else 1.0)
                        if best is None or cost < best["cost"]:
                            best = {"cost": cost, "s": s, "t": t, "ta": ta, "tb": tb}
                if best:
                    break
            if best:
                break
        if best is None:                      # No direct route: approach along the dominant axis, via the second Hanan pass.
            s, t = srcs[0], dsts[0]
            best = {"s": s, "t": t, "ta": _ideal_coord(s, positions[a], positions[b], corner, port_anchor),
                    "tb": _ideal_coord(t, positions[b], positions[a], corner, port_anchor)}
        plan.append({"a": a, "b": b, "cnt": cnt, "mean": r.get("mean_sec", np.nan), **best})
    t2 = time.time()

    # ================= port spreading: same sides shared by multiple edges
    groups = defaultdict(list)
    for i, p in enumerate(plan):
        groups[(p["a"], p["s"])].append((i, "ta", p["b"]))
        groups[(p["b"], p["t"])].append((i, "tb", p["a"]))
    for (node, side), lst in groups.items():
        x, y, w, h = positions[node]
        lo, hi = ((x + corner, x + w - corner) if side in "NS" else (y + corner, y + h - corner))
        if node in loop_rects and side == loop_side[node]:
            hi = min(hi, loop_rects[node][0] - gap * 0.5)
        ax = 0 if side in "NS" else 1
        vals = [plan[i][k] + 1e-4 * ctr(partner)[ax] for i, k, partner in lst]
        for (i, k, _), t in zip(lst, r4_spread(vals, lo, max(lo, hi), gap)):
            plan[i][k] = t

    # ---- door pre-assignment: strip allocation based on thickness
    door_users = defaultdict(list)
    for i, p in enumerate(plan):
        a, b = p["a"], p["b"]
        if zone[a] == zone[b] or not geo:
            continue
        pa = _port_xy(positions[a], p["s"], p["ta"])
        pb = _port_xy(positions[b], p["t"], p["tb"])
        L = max(arrow_len * 1.2, margins[0] + 3.0)
        A, B = _stub_xy(pa, p["s"], L), _stub_xy(pb, p["t"], L)
        dA, dB = _pick_doors(A, B, zone[a], zone[b], geo, {}, door_off)
        for d in (dA, dB):
            if d:
                ax = 0 if d["side"] in "NS" else 1
                door_users[d["id"]].append((i, (A[ax] + B[ax]) / 2.0, d))
    for did, lst in door_users.items():
        d = lst[0][2]
        offs, ws, sc = r4_assign_lanes([(i, width_of(plan[i]["cnt"]), o) for i, o, _ in lst], d, 2.5)
        for i, _o, _d in lst:
            lane_off[(i, did)] = offs[i]
            lane_w[i] = min(lane_w.get(i, 1e9), ws[i])
        log.append(f"door {did}: {len(lst)} lines" + (f" | thickness x{sc:.2f} reduced" if sc < 1 else " | fits"))

    # ================= 2. TRANSITION: final port routing (Hanan only if necessary)
    used_doors, failed = {}, []
    for i, p in enumerate(plan):
        cur["i"] = i
        a, b = p["a"], p["b"]
        res = None
        for use_forb in (True, False):
            res = route_edge(a, p["s"], p["ta"], b, p["t"], p["tb"], True, used_doors, use_forb)
            if res:
                break
        if res is None:                       # last resort: other sides (with ideal ports) and Hanan pass
            for s in "NSWE":
                for t in "NSWE":
                    ta = _ideal_coord(s, positions[a], positions[b], corner, port_anchor)
                    tb = _ideal_coord(t, positions[b], positions[a], corner, port_anchor)
                    r2 = route_edge(a, s, ta, b, t, tb, True, used_doors, False)
                    if r2 and (res is None or r2[1] < res[1]):
                        res = r2
                        p.update({"s": s, "t": t, "ta": ta, "tb": tb})
        p["pts"] = res[0] if res else None
        if res is None:
            failed.append((a, b))
    t3 = time.time()

    # ================= strip separation (parallel, non-overlapping)
    ok = [p for p in plan if p["pts"] is not None]
    routes = [p["pts"] for p in ok]
    hcache = {}

    def hard_for(ri):
        key = frozenset(ancs[ok[ri]["a"]] | ancs[ok[ri]["b"]])
        if key not in hcache:
            parts = [_inf(box_raw, hard_clear), _inf(wall_raw, wall_m)]
            if len(cont_raw):
                mk = np.array([k not in key for k in cont_keys])
                parts.append(_inf(cont_raw[mk], hard_clear))
            hcache[key] = np.vstack([q for q in parts if len(q)])
        return hcache[key]

    fixed_arr = _arr(fixed_rects)
    pre_exact, pre_near = r4_measure(routes, gap * 0.5) if routes else (0.0, 0.0)
    moved, sep_stats = r4_separate(routes, hard_for, fixed_arr, gap, min_end, iters=sep_iters) if routes else (0, {})
    for p, r in zip(ok, routes):
        p["pts"] = r
    exact, near = r4_measure(routes, gap * 0.5) if routes else (0.0, 0.0)
    if routes:
        log.extend(r4_overlap_report(routes, fixed_arr, gap, geo, wt))
        log.extend(r4_top_overlaps(routes, [(p["a"], p["b"]) for p in ok], fixed_arr))
        log.append(f"Separation obstacles: {sep_stats}")
    t4 = time.time()

    # ================= drawing
    def esc(v):
        return html.escape(str(v), quote=True)

    edges_svg, labels, out_tot, in_tot = [], [], defaultdict(int), defaultdict(int)
    for pi, p in enumerate(plan):
        a, b, cnt = p["a"], p["b"], p["cnt"]
        out_tot[a] += cnt
        in_tot[b] += cnt
        w = min(width_of(cnt), lane_w.get(pi, 1e9))
        has_perf = (mode == "perf") and pd.notna(p.get("mean"))
        color = duration_to_color(p["mean"], vmin, vmax) if has_perf else node_colors.get(a, "#5D6D7E")
        pts, extra = p["pts"], ""
        if pts is None:
            pa = _port_xy(positions[a], p["s"], p["ta"])
            pb = _port_xy(positions[b], p["t"], p["tb"])
            pts = _simplify_points(_orthogonalize(manhattan_fallback(pa, pb)))
            extra = ' stroke-dasharray="6 4"'
        p["final"] = pts
        trimmed, poly = _arrow_for(pts, w, st)
        d = f"M {trimmed[0][0]:.1f},{trimmed[0][1]:.1f} " + " ".join(f"L {x:.1f},{y:.1f}" for x, y in trimmed[1:])
        lx, ly, anchor = _label_anchor(pts, w, st)
        lbl = _label_svg(lx, ly, anchor, fmt_dur(p["mean"]) if has_perf else f"{cnt:,}", st)
        tip_extra = f" | avg. delay {fmt_dur(p['mean'])}" if has_perf else ""
        lbl_in_edge = lbl.replace("<text ", '<text class="lbl" ', 1)
        body = (f'<path d="{d}" fill="none" stroke="{color}" stroke-width="{w}"{extra} stroke-linejoin="miter" '
                f'stroke-linecap="butt" opacity="0.9"/><polygon points="{poly}" fill="{color}" opacity="0.9"/>')
        edges_svg.append(f'<g class="e" data-s="{esc(a)}" data-t="{esc(b)}">'
                         f'<title>{esc(a)} -> {esc(b)}\n{cnt:,} transitions{tip_extra}</title>{body}{lbl_in_edge}</g>')
        labels.append((cnt, a, b, lbl))
    for c, n in loops.items():
        lcol = (duration_to_color(loop_perf[c], vmin, vmax)
                if mode == "perf" and c in loop_perf else node_colors.get(c, "#5D6D7E"))
        lp, la, ll = draw_self_loop_svg_side(loop_bb[c], n, lcol, st, width_of(n),
                                             loop_side[c], loop_label(c, n))
        edges_svg.append(f'<g class="e" data-s="{esc(c)}" data-t="{esc(c)}"><title>{esc(c)} -> {esc(c)}\n{n:,}</title>'
                         f'{lp}{la}</g>')
        labels.append((n, c, c, ll))
    shown = sorted([t for t in labels if t[1] != t[2]], key=lambda t: -t[0])
    shown = shown if label_top_n is None else shown[:int(label_top_n)]
    tl = [t[3].replace("<text ", f'<text class="tl" data-s="{esc(t[1])}" data-t="{esc(t[2])}" ', 1)
          for t in shown + [t for t in labels if t[1] == t[2]]]
    nodes_svg = []
    for c, (x, y, w, h) in positions.items():
        nm = (node_names or {}).get(c, "")
        tip = f"{c}" + (f" - {nm}" if nm else "") + f"\nOut: {out_tot.get(c, 0):,} | In: {in_tot.get(c, 0):,}"
        nodes_svg.append(f'<rect class="n" data-id="{esc(c)}" x="{x:.1f}" y="{y:.1f}" width="{w:.1f}" '
                         f'height="{h:.1f}" rx="4" fill="transparent"><title>{esc(tip)}</title></rect>')

    # ================= VALIDATION
    good = [p for p in plan if p["pts"] is not None]
    nonortho = sum(1 for p in good for q, r_ in zip(p["final"], p["final"][1:])
                   if _sdir(q, r_) is None and math.dist(q, r_) > 1e-3)
    leaf_viol = [(p["a"], p["b"]) for p in good if box_crossings(p["final"], leaf_rects)]
    wall_bad = [(p["a"], p["b"]) for p in good if count_wall_crossings(p["final"], walls) > 0]
    door_cross = sum(1 for p in good if zone[p["a"]] != zone[p["b"]])
    nb = [_path_cost(p["final"], p["s"], _OPP[p["t"]])[1] for p in good]
    bend_hist = [sum(1 for b_ in nb if (b_ == k if k < 3 else b_ >= 3)) for k in range(4)]
    if wall_bad:
        log.append(f"Wall violations: {wall_bad[:10]}")
    if leaf_viol:
        log.append(f"Routes crossing boxes: {leaf_viol[:10]}")
    if failed:
        log.append(f"{len(failed)} edges have no box/wall-free path (dashed L): {failed[:5]}")
    log.append(f"Bend distribution (0/1/2/3+): {bend_hist}")
    log.append(f"Strip separation: {moved} segments | overlapping {pre_exact:.0f}px -> {exact:.0f}px | "
               f"near parallel {pre_near:.0f}px -> {near:.0f}px")
    log.append(f"TIME: 1st pass {t2 - t1:.1f}s | 2nd pass {t3 - t2:.1f}s | strip separation {t4 - t3:.1f}s | "
               f"total {time.time() - T0:.1f}s")

    tf = title_font if title_font is not None else st["title_font"] * 2.0
    title_svg = (f'<text x="{tf:.0f}" y="{tf * 1.3:.0f}" fill="black" font-family="Arial" font-size="{tf:.0f}" '
                 f'font-weight="bold">{html.escape(title)}</text>' if title else "")
    legend_svg = ""
    lf = legend_font if legend_font is not None else st["font_size"] * 1.8
    lx0, ly0 = tf, tf * 2.3
    if mode == "perf":
        legend_svg = (
            '<defs><linearGradient id="gperf" x1="0" x2="1" y1="0" y2="0">'
            '<stop offset="0" stop-color="#2ecc71"/><stop offset=".5" stop-color="#f1c40f"/>'
            '<stop offset="1" stop-color="#e74c3c"/></linearGradient></defs>'
            f'<text x="{lx0:.0f}" y="{ly0 + lf:.0f}" fill="black" font-family="Arial" font-size="{lf:.0f}">'
            f'Line Color = avg. delay · Line Width = frequency · '
            f'Hover: highlight links · Scroll: zoom · Drag: pan · Double-Click: reset</text>'
            f'<rect x="{lx0:.0f}" y="{ly0 + lf * 2:.0f}" width="200" height="{lf:.0f}" fill="url(#gperf)"/>'
            f'<text x="{lx0:.0f}" y="{ly0 + lf * 4:.0f}" fill="black" font-family="Arial" font-size="{lf:.0f}">{fmt_dur(vmin)}</text>'
            f'<text x="{lx0 + 200:.0f}" y="{ly0 + lf * 4:.0f}" fill="black" font-family="Arial" font-size="{lf:.0f}" '
            f'text-anchor="end">{fmt_dur(vmax)}</text>'
        )
    else:
        legend_svg = (
            f'<text x="{lx0:.0f}" y="{ly0 + lf:.0f}" fill="black" font-family="Arial" font-size="{lf:.0f}">'
            f'Line Color = frequency (log scale) · Line Width = transition · '
            f'Hover: highlight links · Scroll: zoom · Drag: pan · Double-Click: reset</text>'
        )

    overlay = "\n".join([f"<style>{_CSS_FLOORPLAN}</style>", title_svg, legend_svg, '<g id="edges">'] + edges_svg +
                        ['</g><g id="nodes">'] + nodes_svg + ['</g><g id="tlabels" pointer-events="none">'] + tl +
                        ['</g><g id="hl"></g>', "<script><![CDATA[" + _JS_FLOORPLAN + "]]></script>"])
    content = _make_svg_responsive(content)
    end = content.rfind("</svg>")
    svg = content[:end] + overlay + "\n" + content[end:] if end >= 0 else content
    info = {
        "n_edges": len(df_), "n_loops": len(loops), "failed": len(failed),
        "door_cross": door_cross, "leaf_viol": len(leaf_viol), "wall_viol": len(wall_bad),
        "nonortho": nonortho, "lane_moves": moved, "overlap_exact_px": round(exact),
        "bends": bend_hist, "seconds": round(time.time() - T0, 1), "log": log,
    }
    return svg, info


### A3. Configuration Classes

In [5]:
# ============================================================================
# SINGLE SOURCE OF TRUTH for the column mapping. Every function takes `cfg`.
# frozen=True protects against accidental mutation.
# ============================================================================

@dataclass(frozen=True)
class HierLevel:
    code_col: str
    name_col: str | None
    label: str


@dataclass(frozen=True)
class EventLogConfig:
    case_id_col: str
    activity_code_col: str
    activity_name_col: str | None
    timestamp_col: str | None          # point-event log if not None
    start_timestamp_col: str | None    # interval-event log (Start/Finish) if not None
    end_timestamp_col: str | None
    hierarchy_levels: tuple[HierLevel, ...]

    @property
    def has_interval(self) -> bool:
        """True if each row has both a start and an end time (durations can be computed)."""
        return self.start_timestamp_col is not None and self.end_timestamp_col is not None

    @property
    def ts_col(self) -> str | None:
        """The timestamp used to order operations inside an order."""
        return self.timestamp_col if self.timestamp_col else self.start_timestamp_col

    @property
    def hier_code_cols(self) -> tuple[str, ...]:
        return tuple(lvl.code_col for lvl in self.hierarchy_levels)
    

### A4. Data loading and preparation

In [6]:
def load_event_log(file_bytes: bytes, file_name: str, sheet_name=0) -> pd.DataFrame:
    """CSV or XLSX raw text (dtype=str) is read as-is. Column meanings
    are not yet known — this is resolved by prepare_dataframe() with the cfg."""
    if file_name.lower().endswith(".csv"):
        return pd.read_csv(io.BytesIO(file_bytes), dtype=str)
    if file_name.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(io.BytesIO(file_bytes), sheet_name=sheet_name, dtype=str, engine="openpyxl")
    raise ValueError(f"Unsupported file type: {file_name!r} (only .csv / .xlsx)")


def prepare_dataframe(raw: pd.DataFrame, cfg: EventLogConfig) -> pd.DataFrame:
    """Validates required columns, sets types and keeps only operation rows.

    Rows that do not carry an operation (e.g. Set Order / Update Order / Request Order Complete rows,
    which have no work center, station or start time) are dropped here automatically."""
    df = raw.copy()

    required = [cfg.case_id_col, cfg.activity_code_col] + list(cfg.hier_code_cols)
    required += [cfg.start_timestamp_col, cfg.end_timestamp_col] if cfg.has_interval else [cfg.timestamp_col]
    missing = [c for c in required if c and c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}\nAvailable columns: {list(df.columns)}")

    ts_cols = [c for c in (cfg.timestamp_col, cfg.start_timestamp_col, cfg.end_timestamp_col)
               if c and c in df.columns]
    for col in ts_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    key_cols = [cfg.case_id_col, cfg.activity_code_col, *cfg.hier_code_cols, cfg.ts_col]
    n0 = len(df)
    df = df.dropna(subset=key_cols).reset_index(drop=True)
    if n0 - len(df):
        print(f"{n0 - len(df):,} rows without operation info were dropped (non-operation events / empty fields).")

    str_cols = list(cfg.hier_code_cols) + [cfg.activity_code_col]
    str_cols += [lvl.name_col for lvl in cfg.hierarchy_levels if lvl.name_col]
    if cfg.activity_name_col:
        str_cols.append(cfg.activity_name_col)
    for col in str_cols:
        if col in df.columns:
            df[col] = df[col].astype(str)

    for col in list(cfg.hier_code_cols) + [cfg.activity_code_col]:
        df[col] = df[col].astype("category")
    return df


def safe_sort(frame: pd.DataFrame, cols) -> pd.DataFrame:
    if isinstance(cols, str):
        cols = [cols]
    if len(frame) == 0:
        return frame.reset_index(drop=True)
    keys = [frame[c].to_numpy() for c in reversed(cols)]
    order = np.lexsort(keys)
    return frame.iloc[order].reset_index(drop=True)


def format_duration(seconds):
    seconds = abs(seconds)
    if seconds < 60:
        return f"{seconds:.0f} seconds"
    elif seconds < 3600:
        return f"{seconds/60:.0f} minutes"
    elif seconds < 86400:
        return f"{seconds/3600:.1f} hours"
    else:
        return f"{seconds/86400:.1f} days"


### A5. Floor Plan Overlay

In [7]:
# ============================================================================
# Floor plan (Figma SVG) renderer — draws flow arrows on the floor plan — ONE diagram per hierarchy level.
# The number of levels is taken from cfg.hierarchy_levels (0..N).
# Area boxes are found first from the DOM (label text + containment depth);
# fallback mapping is attempted for unmatched ones using ID hints. The routing engine
# (door/wall aware, orthogonal, grid-based) is implemented in floorplan_routing.py.
# ============================================================================


@dataclass
class FloorPlanSettings:
    # --- fallback (ID-based) mapping
    area_hint: str = "area"                        # a common word in the ID of the area rectangle
    level_hints: tuple[tuple[str, ...], ...] = ()  # in the SAME order as cfg.hierarchy_levels
    # --- Flows will be drawn
    top_n_edges: int | None = None                 # None = all edges; int = most frequent N edges
    min_edge_count: int = 1
    self_loop_min_count: int = 1
    label_top_n: int = 40                          # The most frequent N edge labels to always display
    # --- scale and factory limit
    px_to_m: float = 0.06                          # 1 px = 6 cm in SVG Plan (Figma) coordinates
    wall_thickness: float = 8.0                    # Wall thickness (px)
    zones: dict = field(default_factory=dict)      # {name: {"border_id": str, "door_ids": [str, ...]}}
    # --- DOM area detection / style
    dom_min_size: float = 12.0                     # (px) Boxes smaller than this area are filtered out.
    dom_max_canvas_frac: float = 0.6               # The area of ​​boxes (background) larger than this proportion of the canvas is eliminated.
    ref_stat: str = "max"                          #  line/arrow size reference: max | p90 | median
    margin_px: float | None = None                 # preferred line-to-box distance (px); None = automatic
    box_clearance: float | None = None             # HARD minimum line-to-box distance (px); router never goes below it (None = 1.5)
    loop_clearance: float | None = None            # free space (px) required around a self-loop; None = automatic
    line_gap: float | None = None                  # spacing between parallel lines (px); None = automatic
    sep_iters: int = 5                             # number of overlap-separation passes
    side_search: str = "all"                       # "all": pick box sides by real route cost | "quadrant": old heuristic
    port_anchor: str = "nearest"                   # "nearest": port at side point closest to the other box | "center"


def default_level_hints(label: str) -> tuple[str, ...]:
    """Derives SVG ID hints from the level label (e.g., 'Work Center'): 
    such as 'work center', 'workcenter', 'work_center', 'wc'."""
    words = re.findall(r"[A-Za-z0-9]+", label)
    cand = [label.lower(), "".join(words).lower(), "_".join(words).lower()]
    if len(words) > 1:
        cand.append("".join(w[0] for w in words).lower())
    out, seen = [], set()
    for c in cand:
        n = re.sub(r"[^a-z0-9]", "", c)
        if n and n not in seen:
            seen.add(n)
            out.append(c)
    return tuple(out)


def sanitize_svg(content: str) -> str:
    """Sanitizes the loaded SVG before embedding it in an iframe: script,
    foreignObject, on* event attributes and javascript: links."""
    content = re.sub(r"<script\b.*?</script\s*>", "", content, flags=re.DOTALL | re.IGNORECASE)
    content = re.sub(r"<script\b[^>]*/>", "", content, flags=re.IGNORECASE)
    content = re.sub(r"<foreignObject\b.*?</foreignObject\s*>", "", content, flags=re.DOTALL | re.IGNORECASE)
    content = re.sub(r"\son\w+\s*=\s*(\"[^\"]*\"|'[^']*')", "", content, flags=re.IGNORECASE)
    content = re.sub(r"(href\s*=\s*[\"'])\s*javascript:[^\"']*", r"\1#", content, flags=re.IGNORECASE)
    return content


# ----------------------------------------------------------------------------
# Level transitions ("From-To") matrix
# ----------------------------------------------------------------------------
def build_level_transition_matrix(ops: pd.DataFrame, cfg, code_col: str, name_col: str | None = None):
    """Sequential events with the same level values are counted once; only
    actual movements between different locations are counted (SLP 'From-To' table).
    Returns: (matrix_df, pairs_df[from,to,count(,from_name,to_name)], name_lookup)."""
    ts_col = cfg.ts_col
    cols = [cfg.case_id_col, code_col, ts_col] + ([name_col] if name_col else [])
    frame = ops[cols].dropna(subset=[code_col, ts_col]).copy()
    frame[code_col] = frame[code_col].astype(str)
    frame = safe_sort(frame, [cfg.case_id_col, ts_col])
    name_lookup = {}
    if name_col:
        name_lookup = frame.drop_duplicates(code_col).set_index(code_col)[name_col].astype(str).to_dict()
    frame["prev_code"] = frame.groupby(cfg.case_id_col, sort=False)[code_col].shift(1)
    moves = frame[frame["prev_code"].notna() & (frame["prev_code"] != frame[code_col])]
    if moves.empty:
        empty = pd.DataFrame(columns=["from", "to", "count"])
        return pd.DataFrame(), empty, name_lookup
    pairs = (moves.groupby(["prev_code", code_col]).size().rename("count").reset_index()
                  .rename(columns={"prev_code": "from", code_col: "to"}))
    pairs = pairs.iloc[np.argsort(-pairs["count"].to_numpy(), kind="stable")].reset_index(drop=True)
    codes = sorted(set(pairs["from"]) | set(pairs["to"]))
    matrix = pd.DataFrame(0, index=codes, columns=codes, dtype=int)
    for r in pairs.itertuples(index=False):
        matrix.loc[r[0], r[1]] = r[2]
    if name_lookup:
        pairs["from_name"] = pairs["from"].map(name_lookup)
        pairs["to_name"] = pairs["to"].map(name_lookup)
    return matrix, pairs, name_lookup


def build_level_transition_perf(ops: pd.DataFrame, cfg, code_col: str):
    """Average waiting time (in seconds) for transitions between levels.
    Waiting time = End of previous operation -> Start of next operation (if Start/Finish is available),
    otherwise the difference between events. Negative (overlapping) gaps are clipped to 0.
    Returns: (pairs_perf[from,to,mean_sec,min_sec,max_sec], {code: self-loop average time})"""
    ts_s = cfg.ts_col
    ts_e = cfg.end_timestamp_col if cfg.has_interval else None
    cols = [cfg.case_id_col, code_col, ts_s] + ([ts_e] if ts_e else [])
    fr = ops[cols].dropna(subset=[code_col, ts_s]).copy()
    fr[code_col] = fr[code_col].astype(str)
    fr = safe_sort(fr, [cfg.case_id_col, ts_s])
    g = fr.groupby(cfg.case_id_col, sort=False)
    fr["prev_code"] = g[code_col].shift(1)
    prev_ref = g[ts_s].shift(1)
    if ts_e:
        prev_ref = g[ts_e].shift(1).fillna(prev_ref)
    fr["_gap"] = (fr[ts_s] - prev_ref).dt.total_seconds().clip(lower=0)
    mv = fr[fr["prev_code"].notna() & fr["_gap"].notna()]
    cross = mv[mv["prev_code"] != mv[code_col]]
    same = mv[mv["prev_code"] == mv[code_col]]
    perf = (cross.groupby(["prev_code", code_col])["_gap"]
                 .agg(mean_sec="mean", min_sec="min", max_sec="max").reset_index()
                 .rename(columns={"prev_code": "from", code_col: "to"}))
    loop_perf = same.groupby(code_col)["_gap"].mean().to_dict()
    return perf, loop_perf


def compute_self_loop_counts(ops: pd.DataFrame, cfg, code_col: str) -> dict:
    """Events with the same code occurring consecutively within a case."""
    ts_col = cfg.ts_col
    frame = ops[[cfg.case_id_col, code_col, ts_col]].dropna(subset=[code_col, ts_col]).copy()
    frame[code_col] = frame[code_col].astype(str)
    frame = safe_sort(frame, [cfg.case_id_col, ts_col])
    frame["prev_code"] = frame.groupby(cfg.case_id_col, sort=False)[code_col].shift(1)
    loops = frame[frame["prev_code"].notna() & (frame["prev_code"] == frame[code_col])]
    return loops.groupby(code_col).size().to_dict()


# ----------------------------------------------------------------------------
# Fallback: SVG ID <-> code mapping (for codes not found via DOM tag)
# ----------------------------------------------------------------------------
def _norm(s) -> str:
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())


def match_area_ids_for_level(svg_ids, hints, area_hint, codes):
    """Returns: ({code: svg_id}, unmatched_codes). First exact matches, then (for remaining
    codes) fuzzy matches — order dependency not considered."""
    area_n = _norm(area_hint)
    hints_n = sorted({_norm(h) for h in hints if _norm(h)}, key=len, reverse=True)
    candidates = [i for i in svg_ids if area_n and area_n in _norm(i)]
    hinted = [i for i in candidates if any(h in _norm(i) for h in hints_n)]
    pool = hinted if hinted else candidates
    norm_codes = {_norm(c): c for c in codes}

    def core_of(svg_id):
        core = _norm(svg_id).replace(area_n, "")
        for h in hints_n:
            core = core.replace(h, "")
        return core

    result, used_ids = {}, set()
    for svg_id in pool:                          # 1) exact match
        core = core_of(svg_id)
        if core and core in norm_codes and norm_codes[core] not in result:
            result[norm_codes[core]] = svg_id
            used_ids.add(svg_id)
    for svg_id in pool:                          # 2) fuzzy match (core substring)
        if svg_id in used_ids:
            continue
        core = core_of(svg_id)
        if not core:
            continue
        for cn, orig in norm_codes.items():
            if orig not in result and cn and (cn in core or core in cn):
                result[orig] = svg_id
                used_ids.add(svg_id)
                break
    missing = [c for c in codes if c not in result]
    return result, missing


# ----------------------------------------------------------------------------
# Color Assignment
# ----------------------------------------------------------------------------
def assign_node_colors(codes, saturation=0.65, value=0.85):
    codes = sorted(set(codes))
    n = max(1, len(codes))
    colors = {}
    for i, code in enumerate(codes):
        r, g, b = colorsys.hsv_to_rgb(i / n, saturation, value)
        colors[code] = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"
    return colors


def recolor_floor_plan_areas(svg_content, id_map, node_colors, fill_opacity=0.30, stroke_width=2.5):
    """Paint the area boxes according to node color (existing fill/stroke is deleted)."""
    content = svg_content
    for code, svg_id in id_map.items():
        color = node_colors.get(code)
        if not color or not svg_id:        # Boxes without ID can be in DOM mapping -> not painted
            continue
        pattern = re.compile(rf'(<[a-zA-Z]+\b[^>]*\bid="{re.escape(svg_id)}"[^>]*?)(/?>)')
        m = pattern.search(content)
        if not m:
            continue
        tag_open, tag_close = m.group(1), m.group(2)
        tag_open = re.sub(r'\sfill="[^"]*"', "", tag_open)
        tag_open = re.sub(r'(\sstyle=")([^"]*)(")',
                          lambda mm: mm.group(1) + re.sub(r'fill\s*:\s*[^;]+;?', '', mm.group(2)) + mm.group(3),
                          tag_open)
        tag_open = re.sub(r'\sstroke="[^"]*"', "", tag_open)
        tag_open = re.sub(r'\sstroke-width="[^"]*"', "", tag_open)
        new_open = (f'{tag_open} fill="{color}" fill-opacity="{fill_opacity}" '
                    f'stroke="{color}" stroke-width="{stroke_width}" stroke-opacity="0.9"')
        content = content[:m.start()] + new_open + tag_close + content[m.end():]
    return content


# ----------------------------------------------------------------------------
# SINGLE ENTRY POINT — floor plan diagram for each hierarchy level
# ----------------------------------------------------------------------------
def build_floor_plan_views(svg_content: str, 
                           df: pd.DataFrame, 
                           cfg:EventLogConfig, 
                           fp: FloorPlanSettings, 
                           mode: str = "freq") -> dict:
    """Returns: {"levels": [{label, level_idx, svg, info, centers, positions, unmatched}],
    "geo": factory wall/door geometry, "px_to_m": float, "warnings": [...], "log": [...]}"""
    svg_content = sanitize_svg(svg_content)
    nodes, id_bboxes, _canvas = parse_svg_dom(svg_content, fp.dom_min_size, fp.dom_max_canvas_frac)
    ops = df
    warnings, log, levels = [], [], []

    geo, geo_warn = build_factory_geometry(id_bboxes, fp.zones, fp.wall_thickness)
    warnings += geo_warn
    for name, g in geo.items():
        log.append(f"Zone {name}: doors={[d['id'] for d in g['doors']]} | wall segments={len(g['walls'])}")

    level_codes = [ops[lvl.code_col].astype(str).unique().tolist() for lvl in cfg.hierarchy_levels]

    def _id_fallback(lvl_idx, missing):
        label = cfg.hierarchy_levels[lvl_idx].label
        hints = fp.level_hints[lvl_idx] if lvl_idx < len(fp.level_hints) else default_level_hints(label)
        id_map, _ = match_area_ids_for_level(list(id_bboxes), hints, fp.area_hint, missing)
        return ({c: id_bboxes[s] for c, s in id_map.items() if s in id_bboxes}, dict(id_map))

    pos_by_level, ids_by_level, missing_by_level = dom_positions(nodes, level_codes, _id_fallback, log)
    all_level_positions = {(i, code): box for i, level_positions in enumerate(pos_by_level)
                           for code, box in level_positions.items()}
    shared_style = derive_level_style(all_level_positions, ref_stat=fp.ref_stat)
    shared_title_font = shared_style["title_font"] * 2.0
    shared_legend_font = shared_style["font_size"] * 1.6

    for i, lvl in enumerate(cfg.hierarchy_levels):
        positions, id_map, missing = pos_by_level[i], ids_by_level[i], missing_by_level[i]
        if not positions:
            warnings.append(f"{lvl.label}: No field matched in the SVG (label text and ID hints were tried; "
                            f"area keyword: '{fp.area_hint}').")
            continue
        if missing:
            warnings.append(f"{lvl.label}: {len(missing)} codes did not match in the SVG: "
                            f"{missing[:10]}{' ...' if len(missing) > 10 else ''}")
        no_id = [c for c in positions if not id_map.get(c)]
        if no_id:
            warnings.append(f"{lvl.label}: {len(no_id)} The area does not have an SVG ID; these areas could not be colored. "
                            f"(Flows are mapped out again.).")
        _, pairs, names = build_level_transition_matrix(ops, cfg, lvl.code_col, lvl.name_col)
        perf_df, loop_perf = (build_level_transition_perf(ops, cfg, lvl.code_col)
                              if mode == "perf" else (None, None))
        colors = assign_node_colors(positions.keys())
        # do not paint in perf mode: edge colors will carry duration scalar
        recolored = recolor_floor_plan_areas(svg_content, id_map, colors) if mode == "freq" else svg_content
        containers = {j: pos_by_level[j] for j in range(i)}
        suffix = " — Average Waiting Time" if mode == "perf" else ""
        try:
            t=time.time()
            svg, info = build_floor_plan_overlay(
                recolored, 
                positions, 
                pairs,
                title=f"{lvl.label} Flow (On Floor Plan){suffix}",
                node_colors=colors,
                self_loop_counts=compute_self_loop_counts(ops, cfg, lvl.code_col),
                factory_geo=geo, 
                container_positions=containers, 
                node_names=names,
                wall_t=fp.wall_thickness, 
                min_count=fp.min_edge_count, 
                top_n=fp.top_n_edges,
                label_top_n=fp.label_top_n, 
                margin_px=fp.margin_px, 
                box_clearance=fp.box_clearance,
                loop_clearance=fp.loop_clearance,
                side_search=fp.side_search,
                line_gap=fp.line_gap,
                sep_iters=fp.sep_iters,
                port_anchor=fp.port_anchor,
                ref_stat=fp.ref_stat,
                self_loop_min_count=fp.self_loop_min_count,
                mode=mode, 
                perf_df=perf_df, 
                loop_perf=loop_perf,
                title_font=shared_title_font,
                legend_font=shared_legend_font,
            )
            print(f"Floor plan overlay for {lvl.label} {mode} done in {time.time()-t:.1f}s")
        except ValueError as e:
            warnings.append(f"{lvl.label}: {e}")
            continue
        levels.append({
            "label": lvl.label,
            "level_idx": i,
            "svg": svg,
            "info": info,
            "centers": {c: (x + w / 2, y + h / 2) for c, (x, y, w, h) in positions.items()},
            "positions": positions,
            "unmatched": missing,
            "mode": mode,
        })
    return {"levels": levels, "geo": geo, "px_to_m": fp.px_to_m, "warnings": warnings, "log": log}


### A6. Analysis tables

In [17]:
SLP_LABELS = ("A - Absolutely necessary", "E - Especially important", "I - Important",
              "O - Ordinary closeness OK", "U - Unimportant")


def _duration_sec(ops: pd.DataFrame, cfg) -> pd.Series | None:
    if not cfg.has_interval:
        return None
    return (ops[cfg.end_timestamp_col] - ops[cfg.start_timestamp_col]).dt.total_seconds()


def level_ranking(ops: pd.DataFrame, cfg, level_idx: int, pairs: pd.DataFrame, names: dict) -> pd.DataFrame:
    lvl = cfg.hierarchy_levels[level_idx]
    code = lvl.code_col
    d = ops.copy()
    d[code] = d[code].astype(str)
    dur = _duration_sec(d, cfg)
    if dur is not None:
        d["_dur"] = dur
    g = d.groupby(code, observed=True)
    out = pd.DataFrame({
        "Operation events": g.size(),
        "Distinct orders": g[cfg.case_id_col].nunique(),
        "Distinct operations": g[cfg.activity_code_col].nunique(),
    })
    if dur is not None:
        out["Avg op. duration (sec)"] = round(g["_dur"].mean(), 2)
        out["Avg op. duration"] = out["Avg op. duration (sec)"].map(
            lambda v: format_duration(v) if pd.notna(v) else "-")
    partners = {}
    for a, b in zip(pairs["from"], pairs["to"]):
        partners.setdefault(a, set()).add(b)
        partners.setdefault(b, set()).add(a)
    out[f"Connected to # other {lvl.label}s"] = [len(partners.get(c, ())) for c in out.index]
    out.insert(0, "Name", [names.get(c, "") for c in out.index])
    if level_idx > 0:
        parent = cfg.hierarchy_levels[level_idx - 1].code_col
        d[parent] = d[parent].astype(str)
        out.insert(1, f"Parent {cfg.hierarchy_levels[level_idx - 1].label}",
                   d.groupby(code, observed=True)[parent].agg(lambda s: s.mode().iat[0]))
    out.index.name = lvl.label
    out = out.iloc[np.argsort(-out["Operation events"].to_numpy(), kind="stable")]
    return out.reset_index()


def transfer_time_summary(ops: pd.DataFrame, cfg) -> pd.DataFrame:
    """The gap between consecutive operations (the end of the previous -> the start of the next),
    grouped by the level at which they diverge:
    'Same <inner level>' ... 'Different <outer level>'. The outermost divergence
    (e.g., different work center) is a strong layout signal."""
    levels = cfg.hierarchy_levels
    if not cfg.has_interval or not levels:
        return pd.DataFrame()
    code_cols = list(cfg.hier_code_cols)
    cols = [cfg.case_id_col, cfg.start_timestamp_col, cfg.end_timestamp_col] + code_cols
    seq = ops[cols].dropna(subset=[cfg.start_timestamp_col, cfg.end_timestamp_col]).copy()
    for c in code_cols:
        seq[c] = seq[c].astype(str)
    seq = safe_sort(seq, [cfg.case_id_col, cfg.start_timestamp_col])
    grp = seq.groupby(cfg.case_id_col, sort=False)
    nxt_start = grp[cfg.start_timestamp_col].shift(-1)
    valid = nxt_start.notna().to_numpy().copy()
    gap = (nxt_start - seq[cfg.end_timestamp_col]).dt.total_seconds().to_numpy()
    valid = valid & (gap >= 0)
    n = len(levels)
    first_diff = np.full(len(seq), n)
    for k in reversed(range(n)):
        nxt_code = grp[code_cols[k]].shift(-1).to_numpy()
        first_diff = np.where(seq[code_cols[k]].to_numpy() != nxt_code, k, first_diff)
    first_diff, gap = first_diff[valid], gap[valid]

    def bucket(k):
        if k == n:
            return f"Same {levels[-1].label}"
        if k == 0:
            return f"Different {levels[0].label}"
        return f"Same {levels[k - 1].label}, different {levels[k].label}"

    tmp = pd.DataFrame({"k": first_diff, "gap": gap})
    agg = tmp.groupby("k")["gap"].agg(["count", "mean", "median"])
    agg.columns = ["Moves", "Mean", "Median"]
    res = agg.copy()
    res.insert(0, "Transfer type", [bucket(k) for k in res.index])
    res["Avg wait"] = res["Mean"].map(format_duration)
    res["Median wait (sec)"] = res["Median"]
    return res.drop(columns=["Mean", "Median"]).reset_index(drop=True)


def yearly_overview(ops: pd.DataFrame, cfg) -> pd.DataFrame:
    ts = cfg.ts_col
    d = ops.dropna(subset=[ts]).copy()
    d["year"] = d[ts].dt.year
    agg = {"Operation events": (cfg.case_id_col, "size"),
           "Distinct orders": (cfg.case_id_col, "nunique")}
    for lvl in cfg.hierarchy_levels:
        agg[f"Distinct {lvl.label}"] = (lvl.code_col, "nunique")
    out = d.groupby("year").agg(**agg)
    dur = _duration_sec(d, cfg)
    if dur is not None:
        d["_dur"] = dur
        out["Avg. op duration"] = d.groupby("year")["_dur"].mean().map(format_duration)
    return out


def level_year_share(ops: pd.DataFrame, cfg, level_idx: int) -> pd.DataFrame:
    """Row: code, Column: year, Value: total activity share in that year."""
    code = cfg.hierarchy_levels[level_idx].code_col
    d = ops.dropna(subset=[cfg.ts_col]).copy()
    d["year"] = d[cfg.ts_col].dt.year
    ct = pd.crosstab(d[code].astype(str), d["year"])
    return ct.div(ct.sum(axis=0), axis=1)


def share_change(year_share: pd.DataFrame, year_from: int, year_to: int) -> pd.DataFrame:
    a = year_share[year_from] if year_from in year_share.columns else 0.0
    b = year_share[year_to] if year_to in year_share.columns else 0.0
    ya, yb = str(year_from), str(year_to)
    out = pd.DataFrame({ya: a, yb: b}).fillna(0.0)
    out["Change (pp)"] = (out[yb] - out[ya]) * 100
    return out.iloc[np.argsort(-out["Change (pp)"].to_numpy(), kind="stable")].round(2)


def slp_table(pairs: pd.DataFrame, names: dict) -> pd.DataFrame:
    """A->B and B->A flows are combined; based on the distribution of their own level,
    they are rated A/E/I/O/U according to quantiles (0.5/0.75/0.9/0.97).
    (In the notebook, WS rating was incorrectly using WC quantiles;
    here each level uses its own quantiles.)"""
    cols: list[str] = ["A", "A name", "B", "B name", "Combined flow", "SLP rating"]
    if pairs.empty:
        return pd.DataFrame(columns=cols)
    f, t = pairs["from"].to_numpy(), pairs["to"].to_numpy()
    lo, hi = np.where(f < t, f, t), np.where(f < t, t, f)
    sym = (pd.DataFrame({"A": lo, "B": hi, "Combined flow": pairs["count"].to_numpy()})
             .groupby(["A", "B"], as_index=False)["Combined flow"].sum())
    sym = sym.iloc[np.argsort(-sym["Combined flow"].to_numpy(), kind="stable")].reset_index(drop=True)
    v = sym["Combined flow"].to_numpy()
    q = np.quantile(v, [0.5, 0.75, 0.9, 0.97])
    sym["SLP rating"] = np.select([v >= q[3], v >= q[2], v >= q[1], v >= q[0]],
                                  list(SLP_LABELS[:4]), default=SLP_LABELS[4])
    sym["A name"] = sym["A"].map(names)
    sym["B name"] = sym["B"].map(names)
    return pd.DataFrame(sym.loc[:, cols].copy())


def mismatch_table(sym: pd.DataFrame, positions: dict, geo: dict | None = None,
                   px_to_m: float = 1.0) -> pd.DataFrame:
    """Based on the REAL locations in the floor plan: high flow x distant distance =
    the most valuable pairs to relocate. Distance is in meters (px_to_m scale);
    'Walking via doors' measures factory walls only through doors (vertical-horizontal)."""
    if sym.empty or not positions:
        return pd.DataFrame()
    rows = []
    for r in sym.itertuples(index=False):
        a, b = str(r.A), str(r.B)
        if a not in positions or b not in positions:
            continue
        eu_m, walk_m, za, zb = layout_distance(a, b, positions, geo or {}, px_to_m)
        rows.append({
            "A": a, "A name": r[1], "B": b, "B name": r[3],
            "Zone A": za or "outside", "Zone B": zb or "outside",
            "Combined flow": r[4], "SLP rating": r[5],
            "Straight-line (m)": round(eu_m, 1),
            "Walking via doors (m)": round(walk_m, 1),
            "Flow x Walking distance (m)": round(r[4] * walk_m, 1),
        })
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    order = np.argsort(-out["Flow x Walking distance (m)"].to_numpy(), kind="stable")
    return out.iloc[order].reset_index(drop=True)


# ----------------------------------------------------------------------------
# ONLY ONE ENTRY POINT
# ----------------------------------------------------------------------------
def run_analysis(df: pd.DataFrame, cfg: EventLogConfig) -> dict:
    levels = []
    for i, lvl in enumerate(cfg.hierarchy_levels):
        _, pairs, names = build_level_transition_matrix(df, cfg, lvl.code_col, lvl.name_col)
        levels.append({
            "label": lvl.label,
            "level_idx": i,
            "pairs": pairs,
            "ranking": level_ranking(df, cfg, i, pairs, names),
            "year_share": level_year_share(df, cfg, i),
            "slp": slp_table(pairs, names),
        })
    return {
        "transfer": transfer_time_summary(df, cfg),
        "yearly": yearly_overview(df, cfg),
        "levels": levels,
    }


### A7. Helpers

In [10]:
def show_svg(svg: str, height: int = 800):
    """Displays the SVG in an iframe (so that zoom/pan/hover JS works)."""
    doc = "<html><body style='margin:0;background:#fff'>" + svg + "</body></html>"
    display(HTML(f'<iframe srcdoc="{html.escape(doc, quote=True)}" width="100%" height="{height}" '
                 f'style="border:1px solid #ddd"></iframe>'))


def save_text(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print("Saved:")


## Section B — Settings

Modify the column names below to match your file.

Data model: **one operation per row** (Start/Finish time, WorkCenter, WorkStation, Operation).
Rows of other event types (Set Order, Update Order, Request Order Complete) may stay in the file; they have no work center / station / start time and are dropped automatically.

In [ ]:
# ---------------- Folder Settings ----------------
NOTEBOOK_PATH = globals().get("__vsc_ipynb_file__", None)
NOTEBOOK_NAME = os.path.splitext(os.path.basename(NOTEBOOK_PATH))[0] if NOTEBOOK_PATH else "output"
OUTPUT_DIR = NOTEBOOK_NAME + "_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------- Data File ----------------
FILE_PATH = "../../data/log.csv.gz"          # .csv or .xlsx

# ---------------- Column mapping ----------------
cfg = EventLogConfig(
    case_id_col="OrderId",
    activity_code_col="OperationCode",
    activity_name_col="OperationName",           # None if there is no operation name column
    timestamp_col=None,                          # only for point-event logs; None for Start/Finish logs
    start_timestamp_col="StartTime",             # interval-event log (Start/Finish)
    end_timestamp_col="FinishTime",
    hierarchy_levels=(                           # from outside to inside
        HierLevel(code_col="WorkCenterCode",  name_col="WorkCenterName",  label="Workcenter"),
        HierLevel(code_col="WorkStationCode", name_col="WorkStationName", label="Workstation"),
    ),
)

# ---------------- Floor plan ----------------
FLOORPLAN_SVG = "../../data/factory.svg"        # Figma export
fp_settings = FloorPlanSettings(
    area_hint="area",
    level_hints=tuple(default_level_hints(l.label) for l in cfg.hierarchy_levels),
    top_n_edges=None, 
    min_edge_count=1, 
    label_top_n=40,
    px_to_m=0.06,            # 1 SVG px = how many meters in the real world (Figma export scale)
    wall_thickness=8.0,
    zones={
        "Assembly": {"border_id": "AssemblyBorder", "door_ids": ["AssemblyDoor_1", "AssemblyDoor_2"]},
        "Mechanic": {"border_id": "MechanicBorder", "door_ids": ["MechanicDoor_1"]},
    },                       # leave empty {} if there are no zones
    dom_min_size=12.0,
    ref_stat="max",
    margin_px=None,          # preferred line-to-box distance (px). None = automatic (~0.6 x corridor margin)
    box_clearance=5,      # HARD minimum line-to-box distance (px). None = 1.5. e.g. 6.0 -> lines never closer than 6 px
    loop_clearance=None,     # free space around a self-loop (px). None = automatic
    line_gap=4,           # spacing between parallel lines (px). None = auto (~18 px). Larger -> lines further apart
    sep_iters=10,             # overlap-separation passes; raise (e.g. 15) if overlaps remain
    side_search="all",       # "all": box sides chosen by real route cost | "quadrant": old heuristic
    port_anchor="nearest",   # "nearest": port at the side point closest to the other box | "center": middle of the side
)


## Section C — Run

### C1. Load and prepare data

In [ ]:
with open(FILE_PATH, "rb") as f:
    file_bytes = f.read()
file_name = os.path.basename(FILE_PATH)

raw_df = load_event_log(file_bytes, file_name)
print(f"{len(raw_df):,} rows, {len(raw_df.columns)} columns")
#display(raw_df.head(10))

df = prepare_dataframe(raw_df, cfg)
print(f"{len(df):,} operation rows | {df[cfg.case_id_col].nunique():,} orders | "
      + " | ".join(f"{df[l.code_col].nunique()} {l.label}" for l in cfg.hierarchy_levels))


305,818 rows, 14 columns


,OrderId,OperationCode,OperationName,WorkCenterCode,WorkCenterName,WorkStationCode,WorkStationName,StartTime,FinishTime,timestamp,User,IsFlowEvent,FlowAttachLevel,FlowAttachCode
0,163990,005,Press,009,Machining Department,009-02,Press + Drill + Guillotine Shear,2018-07-07 13:32:00,2018-07-07 13:32:27,2018-07-07 13:32:00,planning_2,False,NaN,NaN
1,192541,030,Aluminium CNC Processing,009,Machining Department,009-07,CNC Machining (Fom),2018-06-10 10:20:00,2018-06-10 10:21:27,2018-06-10 10:20:00,cnc_falcon_operator,False,NaN,NaN
2,193005,030,Aluminium CNC Processing,009,Machining Department,009-04,CNC Machining,2018-06-12 14:11:00,2018-06-12 14:14:27,2018-06-12 14:11:00,cnc_falcon_operator,False,NaN,NaN
3,196036,030,Aluminium CNC Processing,009,Machining Department,009-03,CNC Machining (Falcon),2018-06-25 05:52:00,2018-06-25 05:53:27,2018-06-25 05:52:00,cosmo_cnc,False,NaN,NaN
4,196038,030,Aluminium CNC Processing,009,Machining Department,009-04,CNC Machining,2018-06-12 14:29:00,2018-06-12 14:29:27,2018-06-12 14:29:00,cnc_falcon_operator,False,NaN,NaN
5,197570,016,Hole Drilling,009,Machining Department,009-06,Deburring & Cleaning,2018-07-06 11:11:00,2018-07-06 11:12:27,2018-07-06 11:11:00,cosmo_cnc,False,NaN,NaN
6,202225,001,Glass Cutting,009,Machining Department,009-01,CNC Profile Cutting,2018-06-08 14:26:00,2018-06-08 14:26:27,2018-06-08 14:26:00,purchase_1,False,NaN,NaN
7,202226,002,Aluminium Cutting,009,Machining Department,009-01,CNC Profile Cutting,2018-06-08 14:26:00,2018-06-08 14:26:27,2018-06-08 14:26:00,purchase_1,False,NaN,NaN
8,202229,002,Aluminium Cutting,009,Machining Department,009-01,CNC Profile Cutting,2018-06-08 14:27:00,2018-06-08 14:27:27,2018-06-08 14:27:00,purchase_1,False,NaN,NaN
9,202234,030,Aluminium CNC Processing,009,Machining Department,009-07,CNC Machining (Fom),2018-06-09 10:41:00,2018-06-09 10:41:27,2018-06-09 10:41:00,cosmo_cnc,False,NaN,NaN


151,712 rows without operation info were dropped (non-operation events / empty fields).
154,106 operation rows | 75,611 orders | 23 Workcenter | 47 Workstation


### C2. Process flow on the floor plan — frequency and performance (one diagram per hierarchy level)

In [19]:
with open(FLOORPLAN_SVG, "r", encoding="utf-8", errors="replace") as f:
    svg_text = f.read()

floors = {}
for mode in ("freq", "perf"):
    fl = build_floor_plan_views(svg_text, df, cfg, fp_settings, mode=mode)
    floors[mode] = fl
    if mode == "freq":
        for w in fl["warnings"]:
            print("⚠️", w)
    for lv in fl["levels"]:
        inf = lv["info"]
        print(f"== {lv['label']} [{mode}] == {inf['n_edges']} flow + {inf['n_loops']} self-loop | "
              f"{inf['door_cross']} door crossings | {inf['failed']} unresolved | "
              f"box violations {inf['leaf_viol']}, wall violations {inf['wall_viol']} | {inf['seconds']} sec")
        #show_svg(lv["svg"])
        save_text(f"{OUTPUT_DIR}/{lv['label']}-floorplan-{mode}.svg", lv["svg"])

floor = floors["freq"]      # geometry/positions used by the mismatch table in C3


Floor plan overlay for Workcenter freq done in 0.4s
Floor plan overlay for Workstation freq done in 1.2s
== Workcenter [freq] == 44 flow + 20 self-loop | 21 door crossings | 0 unresolved | box violations 0, wall violations 0 | 0.4 sec
Saved:
== Workstation [freq] == 137 flow + 43 self-loop | 41 door crossings | 0 unresolved | box violations 0, wall violations 0 | 1.1 sec
Saved:
Floor plan overlay for Workcenter perf done in 0.5s
Floor plan overlay for Workstation perf done in 1.3s
== Workcenter [perf] == 44 flow + 20 self-loop | 21 door crossings | 0 unresolved | box violations 0, wall violations 0 | 0.5 sec
Saved:
== Workstation [perf] == 137 flow + 43 self-loop | 41 door crossings | 0 unresolved | box violations 0, wall violations 0 | 1.2 sec
Saved:


### C3. Routing Benchmark

In [ ]:
res = run_benchmark(svg_text, df, cfg, fp_settings,
                    routing_ns=globals(), 
                    views_ns=globals(),
                    build_views=build_floor_plan_views, 
                    repeats=10,
                    outdir= OUTPUT_DIR + "/benchmark_out"
                    )

### C4. Analysis tables

In [20]:
an = run_analysis(df, cfg)

print("Overall Yearly Overview")
display(an["yearly"])
an["yearly"].to_csv(os.path.join(OUTPUT_DIR, "Overall Yearly Overview.csv"))


Overall Yearly Overview


,Operation events,Distinct orders,Distinct Workcenter,Distinct Workstation,Avg. op duration
year,,,,,
2018,8671,4476,7,15,34 minutes
2019,18115,10117,9,18,28 minutes
2020,25546,12305,13,26,1.1 hours
2021,35923,16988,13,28,56 minutes
2022,65851,32024,21,40,42 minutes


In [21]:
if not an["transfer"].empty:
    print("Transfer (waiting) time — by separation level")
    display(an["transfer"])
    an["transfer"].to_csv(os.path.join(OUTPUT_DIR, "Overall Transfer Waiting Time.csv"), index=False)


Transfer (waiting) time — by separation level


,Transfer type,Moves,Avg wait,Median wait (sec)
0,Different Workcenter,5638,3.5 days,67896.0
1,"Same Workcenter, different Workstation",23854,2.8 days,77373.0
2,Same Workstation,43810,1.2 days,12902.0


In [ ]:
years = [int(y) for y in an["yearly"].index]
floor_by_idx = {lv["level_idx"]: lv for lv in floor["levels"]}

for lv in an["levels"]:
    i = lv["level_idx"]
    print("\n" + "=" * 20, lv["label"], "=" * 20)
    print("Ordered by operation events (descending) — first 30")
    display(lv["ranking"].head(30))
    lv["ranking"].head(30).to_csv(os.path.join(OUTPUT_DIR, f"{lv['label']} First 30 Events Descending Ordered by Operation.csv"), index=False)

    if "Avg op. duration (sec)" in lv["ranking"].columns:
        slow = lv["ranking"][lv["ranking"]["Operation events"] > 30]
        print("The Slowest Records Includes More Than 30 Events Descending by AVG Operation Duration")
        display(slow.sort_values("Avg op. duration (sec)", ascending=False).head(10))
        slow.to_csv(os.path.join(OUTPUT_DIR, f"{lv['label']} The Slowest Records Includes More Than 30 Events Descending by AVG Operation Duration.csv"), index=False)

    if len(years) >= 2:
        y_from, y_to = years[0], years[-1]
        sc = share_change(lv["year_share"], y_from, y_to)
        print(f"Share Change From {years[0]} To {years[-1]}")
        display(sc.head(20))
        sc.to_csv(os.path.join(OUTPUT_DIR, f"{lv['label']} Share Change From {years[0]} To {years[-1]}.csv"), index=False)

    print("Systematic Layout Planning (SLP): which pairs should be close?")
    display(lv["slp"].head(30))
    fl = floor_by_idx.get(i)
    if fl is not None:
        mm = mismatch_table(lv["slp"], fl["positions"], floor["geo"], floor["px_to_m"])
        if not mm.empty:
            print("Current layout: Flow × Distance (high flow + far apart = relocation candidate)")
            display(mm.head(30))
            mm.to_csv(os.path.join(OUTPUT_DIR, f"{lv['label']} Mismatch From {years[0]} To {years[-1]}.csv"), index=False)



==================== Workcenter ====================
Ordered by operation events (descending) — first 30


,Workcenter,Name,Operation events,Distinct orders,Distinct operations,Avg op. duration (sec),Avg op. duration,Connected to # other Workcenters
0,009,Machining Department,66755,31250,40,2727.20,45 minutes,11
1,017,Electronics Assembly Department,37193,21209,19,1708.37,28 minutes,4
2,001,Air Duct Assembly Line,20191,9732,14,4711.07,1.3 hours,4
3,006,Body Assembly Department,7171,2857,8,3522.00,59 minutes,3
4,003,SMD Assembly Department,6232,3941,5,2676.96,45 minutes,2
5,028,Shipping & Logistics Department,3484,2827,4,256.82,4 minutes,6
6,020,Plastic Injection Department,3413,1942,5,12577.44,3.5 hours,2
7,024,HANLI Machining Department,2713,1676,9,1104.93,18 minutes,4
8,027,Special Vehicle Assembly Line,1814,979,5,1661.25,28 minutes,3
9,012,Warehouse,1590,1349,10,311.98,5 minutes,6


The Slowest Records Includes More Than 30 Events Descending by AVG Operation Duration


,Workcenter,Name,Operation events,Distinct orders,Distinct operations,Avg op. duration (sec),Avg op. duration,Connected to # other Workcenters
6,020,Plastic Injection Department,3413,1942,5,12577.44,3.5 hours,2
18,033,Main Assembly Line 3,82,78,5,9345.54,2.6 hours,0
2,001,Air Duct Assembly Line,20191,9732,14,4711.07,1.3 hours,4
11,023,Shelving Welding Shop,758,609,2,3801.48,1.1 hours,0
3,006,Body Assembly Department,7171,2857,8,3522.00,59 minutes,3
0,009,Machining Department,66755,31250,40,2727.20,45 minutes,11
16,035,Electronics Manufacturing Line,209,108,7,2684.33,45 minutes,1
4,003,SMD Assembly Department,6232,3941,5,2676.96,45 minutes,2
14,036,Wiring Harness Line,263,185,5,2341.53,39 minutes,0
15,034,Coating Department,212,64,2,1712.46,29 minutes,0


Share Change From 2018 To 2022


,2018,2022,Change (pp)
WorkCenterCode,,,
001,0.00,0.18,17.46
017,0.18,0.26,7.87
028,0.00,0.04,3.77
020,0.00,0.03,2.80
024,0.00,0.02,2.18
012,0.00,0.01,1.29
002,0.00,0.01,1.13
027,0.00,0.01,0.82
031,0.00,0.01,0.59


... (10 outputs left out)

SLP: which pairs should be close?


,A,A name,B,B name,Combined flow,SLP rating
0,009-01,CNC Profile Cutting,009-03,CNC Machining (Falcon),14328,A - Absolutely necessary
1,009-01,CNC Profile Cutting,009-02,Press + Drill + Guillotine Shear,2895,A - Absolutely necessary
2,009-02,Press + Drill + Guillotine Shear,009-03,CNC Machining (Falcon),2546,A - Absolutely necessary
3,001-05,Main Assembly 1,028-01,Final Product Shipping,2380,E - Especially important
4,009-01,CNC Profile Cutting,009-07,CNC Machining (Fom),1819,E - Especially important
5,009-02,Press + Drill + Guillotine Shear,027-02,Auxiliary / Secondary Operation,1569,E - Especially important
6,009-01,CNC Profile Cutting,009-06,Deburring & Cleaning,1386,E - Especially important
7,001-07,Main Assembly 2,028-01,Final Product Shipping,711,E - Especially important
8,009-02,Press + Drill + Guillotine Shear,009-06,Deburring & Cleaning,709,E - Especially important
9,002-07,Outsourced / Subcontracted Operation,009-02,Press + Drill + Guillotine Shear,495,I - Important


Current layout: Flow × Distance (high flow + far apart = relocation candidate)


,A,A name,B,B name,Zone A,Zone B,Combined flow,SLP rating,Straight-line (m),Walking via doors (m),Flow x Walking distance (m)
0,009-01,CNC Profile Cutting,009-03,CNC Machining (Falcon),Mechanic,Mechanic,14328,A - Absolutely necessary,30.3,30.6,438436.8
1,001-05,Main Assembly 1,028-01,Final Product Shipping,Assembly,Assembly,2380,E - Especially important,64.9,81.1,193137.0
2,009-02,Press + Drill + Guillotine Shear,027-02,Auxiliary / Secondary Operation,Mechanic,Assembly,1569,E - Especially important,77.5,109.0,170979.9
3,009-02,Press + Drill + Guillotine Shear,009-03,CNC Machining (Falcon),Mechanic,Mechanic,2546,A - Absolutely necessary,32.1,42.0,106932.0
4,009-01,CNC Profile Cutting,009-07,CNC Machining (Fom),Mechanic,Mechanic,1819,E - Especially important,45.9,46.2,84037.8
5,009-01,CNC Profile Cutting,009-06,Deburring & Cleaning,Mechanic,Mechanic,1386,E - Especially important,44.9,50.9,70572.1
6,001-07,Main Assembly 2,028-01,Final Product Shipping,Assembly,Assembly,711,E - Especially important,62.9,81.8,58166.9
7,006-01,Body Assembly,009-02,Press + Drill + Guillotine Shear,Assembly,Mechanic,279,I - Important,119.0,144.3,40259.7
8,009-01,CNC Profile Cutting,009-02,Press + Drill + Guillotine Shear,Mechanic,Mechanic,2895,A - Absolutely necessary,12.6,13.2,38214.0
9,009-02,Press + Drill + Guillotine Shear,009-06,Deburring & Cleaning,Mechanic,Mechanic,709,E - Especially important,44.3,50.0,35420.1
